# Verifiable Quantum Circuit Compilation

## v7 · DFTHC-SOS Angle Loader Search (stage A)

**Question:** can we lower the 15 chemistry ANGLE loaders while keeping the selected release's words, tree,
factorization and weights fixed? The public chemistry headline moved from **16,233 to 15,777 Toffolis**, largely
through loader recompilation and relation search. The audited receipts establish HW-12 = 15,861, HW-13 = 15,787,
and HW-14 = 15,777; the last step also changed **13 dead angle words**. The 16,233 starting headline is from our
earlier notebook, not an independently recovered HW receipt. We therefore test compatibility rather than assume
that any two public C40 archives have identical tables.

For the paired-product/cache family with `(R,B,C)=(9,143,40)`, `J=5`, `brot=8`, and 89 cache Toffolis per loader,

\[
T_k=89+\frac{525-r_k}{2},\qquad
T_{\rm total}=6741+2\sum_k T_k=17286-\sum_k r_k.
\]

Here `r_k` includes structural coded-bit zeros and must admit a fully paired, phase-compatible decomposition.
**Two usable relations save one Toffoli per forward pass**, and hence two in the chemistry total. This identity is
not a lower bound for every possible compiler. At fixed weights and Q, a saving of `s` per pass changes a 24.17645
portfolio GM by `24.17645*((15777/(15777-2*s))**(1/3)-1)`: about +0.0102 for `s=10`, +0.0205 for `s=20`,
and +0.0616 for `s=60`. These are arithmetic scenarios, not measured outcomes.

**Kaggle setup:** Python 3.13, CPU only, four cores, Internet off, no installs. Attach the inputs in CONFIG below.
The default run uses a 148-minute hard watchdog, a 30-minute full-verification reserve, and a 25-minute four-process
calibration. No competition data or compiler was run locally while authoring this notebook. Live label at authoring:
`v59_c40h14892y5`; a newer label is recorded and checked, not rejected merely for being newer.

The notebook writes `loader_search_A.json`, compiler sources with LICENSE/NOTICE, plans and diagnostics.
It writes `submission.zip` only after a real improvement and three full official PASS results.


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
REQUIRED_COMPETITION = 'google-verifiable-quantum-circuit-compilation'
REQUIRED_DATASETS = [
    'heliosli/qcc-circuits-live',
    'marculera/dfthc-compact-pipeline',
]
OPTIONAL_DATASETS = [
    'matousakura/best-of-public-portfolio',
    'manwithacat/vqcc-apache-public-circuits',
    'manwithacat/vqcc-hedge-wizardry',
    'dmitriigluzdov/vqcc-femoco-public-factorization',
]
# Optional circuits include HW-12/13/14 families and open_circuits_v*;
# missing inputs print an inventory and reduce the reported coverage.
# lightningv08 is embedded below; it needs no dataset or Internet access.
EXPECTED_PORTFOLIO = 'v59_c40h14892y5'
SEED = 20261103
SALT = 'flexonafft-v7-angle-stage-A'
TOTAL_MINUTES = 148                  # includes I/O and verification; <=150 with watchdog cleanup
VERIFICATION_RESERVE_MINUTES = 30    # must remain >=25; no ZIP if short
REPRODUCTION_MINUTES = 20
LERA_HARD_SECONDS = 180              # default shared trials=6, seed=k, one table/job
LIGHTNING_SOFT_SECONDS = 20          # unchanged best_plan defaults, hard cap is +20 seconds
PUBLIC_MINUTES = 25
PUBLIC_JOB_SECONDS = 60
CALIBRATION_MINUTES = 25             # target 20-30 wall minutes across all fifteen tables
CALIBRATION_JOB_SECONDS = 75
REFERENCE_COST = dict(invsqrt=2076774.233715282, dqi=24439463583672.93,
                      dfthcsos=3355599588.3155375)
assert 60 <= TOTAL_MINUTES <= 148
assert VERIFICATION_RESERVE_MINUTES >= 25
assert 20 <= CALIBRATION_MINUTES <= 30


In [ ]:
# ── Session ───────────────────────────────────────────────────────────────────
import os
os.environ.update(OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
# No numpy import occurs before the thread caps, including in spawn workers.
import ast, glob, hashlib, json, signal, subprocess, sys, time
from pathlib import Path
NB_T0 = time.monotonic()
ROOT = Path('/kaggle/working')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
HARD_DEADLINE = NB_T0 + TOTAL_MINUTES*60
for name in ('submission.zip', 'submission.zip.partial', 'provenance.json', 'loader_search_A.json'):
    (ROOT/name).unlink(missing_ok=True)
print('CPU only | BLAS threads: 1 | planner workers: 4 | start method: spawn')
print('Required:', REQUIRED_COMPETITION, *REQUIRED_DATASETS, sep='\n  ')
print('Optional:', *OPTIONAL_DATASETS, sep='\n  ')
DRIVER_PARTS = []


## Audited offline planner sources

Vladislav Lavrentev / lightningv08's v15 `COMPILER_SOURCE` was extracted with literal parsing, without executing
the public notebook. The following cell preserves the audited Python files, Apache-2.0 LICENSE, NOTICE, and
extraction/receipt metadata exactly. The historical receipt is provenance only; it never substitutes for a verifier
run here. Only the planner is used: its older full-cache native emitter needs 140 qubits on C40, above the cap.
Lera's public `tensorqrom.py` is loaded directly from the attached dataset; no build/encode/JAX training stack is imported.


In [ ]:
LIGHTNING_SOURCE = {
  "LICENSE": "\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      \"License\" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      \"Licensor\" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      \"Legal Entity\" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      \"control\" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      \"You\" (or \"Your\") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      \"Source\" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      \"Object\" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      \"Work\" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      \"Derivative Works\" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      \"Contribution\" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, \"submitted\"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as \"Not a Contribution.\"\n\n      \"Contributor\" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a \"NOTICE\" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an \"AS IS\" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets \"[]\"\n      replaced with your own identifying information. (Don't include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same \"printed page\" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the \"License\");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an \"AS IS\" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n",
  "NOTICE": "Derived J5 compiler: Vladislav Lavrentev, 2026.\nBase generic kernel-relation emitter: Ahmed Berat Ozer, public v8 components, Apache 2.0.\nPaired-product kernel, prefix cache and HMR emitter: mrmeows, Apache 2.0, with credited predecessors.\n",
  "borrowed_prefix.py": "#!/usr/bin/env python3\n\"\"\"Organizer-only coherent prefix lookup with restored dirty output helpers.\n\nThe source table is required explicitly. This file imports no fitted data or\nprevious decomposition certificate. It only emits X, CX and CCX gates.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.util\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nimport numpy as np\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef mobius(table):\n    result = np.asarray(table, dtype=np.int64).copy()\n    ids = np.arange(len(result))\n    for bit in range((len(result)-1).bit_length()):\n        rows = ids[(ids & (1 << bit)) != 0]\n        result[rows] ^= result[rows ^ (1 << bit)]\n    return result\n\n\ndef delta_batch(words, size, k):\n    def part(index):\n        result = np.zeros((len(words), size), dtype=np.int64)\n        if index >= 0:\n            start = index * size\n            width = max(0, min(size, words.shape[1] - start))\n            result[:, :width] = words[:, start:start+width]\n        return result\n    return part(k) ^ part(k-1)\n\n\ndef build(delta, brot, clean_cap):\n    import kickmix as km\n    count, size = delta.shape\n    n = (count-1).bit_length()\n    coefficients = mobius(delta)\n    assert np.array_equal(mobius(coefficients), delta)\n    used = set(map(int, np.flatnonzero(np.any(coefficients != 0, axis=1))))\n    for mask in tuple(used):\n        while mask:\n            mask ^= 1 << (mask.bit_length()-1)\n            if mask:\n                used.add(mask)\n    children = {mask: [] for mask in used if mask}\n    for mask in sorted(used):\n        if mask.bit_count() > 1:\n            children[mask ^ (1 << (mask.bit_length()-1))].append(mask)\n\n    builder = km.CircuitBuilder()\n    qubits = list(builder.create_quantum_register(n, 'selection'))\n    for j in range(size):\n        qubits += list(builder.create_quantum_register(brot, f'word_{j}'))\n    target_ids = list(range(n, n + brot*size))\n    scratch_start = len(qubits)\n    qubits += list(builder.alloc_qubits(clean_cap))\n    if len(qubits) > 128:\n        raise ValueError(f'layout requires {len(qubits)} qubits')\n    trace = []\n    fallback = []\n\n    def gate(name, *ids):\n        trace.append((name, *ids))\n        getattr(builder, name.lower())(*[qubits[x] for x in ids])\n\n    def targets(mask):\n        return [n + j*brot + bit for j in range(size)\n                for bit in range(brot) if int(coefficients[mask,j]) >> bit & 1]\n\n    def multi_x(controls, target):\n        k = len(controls)\n        if k <= 2:\n            gate(('X', 'CX', 'CCX')[k], *controls, target)\n            return\n        dirty = [q for q in target_ids if q != target][:k-2]\n        if len(dirty) != k-2:\n            raise ValueError('not enough distinct borrowed output bits')\n        # B = U CCX(c0,c1,d0) U^-1 toggles the last dirty bit by\n        # product(c0,...,c[k-2]). T B T B then toggles only the target.\n        # There are 4*k-8 CCX gates, and every dirty bit is restored.\n        def bridge():\n            for i in reversed(range(1,k-2)):\n                gate('CCX', dirty[i-1], controls[i+1], dirty[i])\n            gate('CCX', controls[0], controls[1], dirty[0])\n            for i in range(1,k-2):\n                gate('CCX', dirty[i-1], controls[i+1], dirty[i])\n        gate('CCX', dirty[-1], controls[-1], target)\n        bridge()\n        gate('CCX', dirty[-1], controls[-1], target)\n        bridge()\n\n    def fan_product(controls, outputs):\n        if not outputs:\n            return\n        pivot, *others = outputs\n        for q in others:\n            gate('CX', pivot, q)\n        multi_x(controls, pivot)\n        for q in others:\n            gate('CX', pivot, q)\n\n    def direct_subtree(mask, ancestor, control):\n        outputs = targets(mask)\n        if outputs:\n            rest = mask ^ ancestor\n            controls = [control] + [bit for bit in range(n) if rest >> bit & 1]\n            fan_product(controls, outputs)\n            fallback.append({'mask': mask, 'controls':len(controls),\n                             'ccx': 1 if len(controls)==2 else 4*len(controls)-8})\n        for child in children[mask]:\n            direct_subtree(child, ancestor, control)\n\n    def visit(mask, control):\n        for q in targets(mask):\n            gate('CX', control, q)\n        for child in children[mask]:\n            new_bit = child.bit_length()-1\n            if not children[child]:\n                fan_product([control, new_bit], targets(child))\n            elif child.bit_count()-1 <= clean_cap:\n                temporary = scratch_start + child.bit_count()-2\n                gate('CCX', control, new_bit, temporary)\n                visit(child, temporary)\n                gate('CCX', control, new_bit, temporary)\n            else:\n                direct_subtree(child, mask, control)\n\n    for q in targets(0):\n        gate('X', q)\n    for bit in range(n):\n        if 1 << bit in used:\n            visit(1 << bit, bit)\n    circuit = builder.finish_circuit()\n    num_ccx = sum(t[0] == 'CCX' for t in trace)\n    assert int(circuit.operation_statistics().num_toffoli_like) == num_ccx\n    return circuit, trace, {'num_qubits':int(circuit.num_qubits),\n        'toffoli_count':num_ccx, 'clean_scratch':clean_cap,\n        'nonlinear_anf_terms':sum(m.bit_count()>1 for m in used),\n        'fallback_products':fallback, 'gates':len(trace)}\n\n\ndef symbolic_check(trace, delta, brot, num_qubits):\n    \"\"\"Exact address-packed affine labels cover every arbitrary target state.\"\"\"\n    count, size = delta.shape\n    n = (count-1).bit_length()\n    all_addresses = (1 << count)-1\n    def packed(bits):\n        return int.from_bytes(np.packbits(np.asarray(bits,dtype=np.uint8),\n                                         bitorder='little').tobytes(), 'little')\n    addresses = np.arange(count, dtype=np.int64)\n    states = [{-1:packed(addresses >> bit & 1)} for bit in range(n)]\n    states += [{i:all_addresses} for i in range(size*brot)]\n    states += [{} for _ in range(num_qubits-len(states))]\n    def add(target, source):\n        for label, bits in source.items():\n            value = states[target].get(label,0) ^ bits\n            if value:\n                states[target][label] = value\n            else:\n                states[target].pop(label,None)\n    for instruction in trace:\n        name, *q = instruction\n        if name == 'X':\n            add(q[0], {-1:all_addresses})\n        elif name == 'CX':\n            add(q[1], states[q[0]])\n        elif name == 'CCX':\n            left, right = states[q[0]], states[q[1]]\n            if any(k >= 0 for k in left):\n                left, right = right, left\n            if any(k >= 0 for k in left):\n                raise AssertionError('unexpected target-quadratic intermediate')\n            control = left.get(-1,0)\n            add(q[2], {key:value & control for key,value in right.items()})\n        else:\n            raise AssertionError(f'non-classical gate {name}')\n    for bit in range(n):\n        assert states[bit] == {-1:packed(addresses >> bit & 1)}\n    for j in range(size):\n        for bit in range(brot):\n            expected = {j*brot+bit:all_addresses}\n            value = packed(delta[:,j] >> bit & 1)\n            if value:\n                expected[-1] = value\n            assert states[n+j*brot+bit] == expected\n    assert all(not s for s in states[n+size*brot:])\n    return {'status':'PASS', 'all_valid_addresses':count,\n            'arbitrary_target_bits':size*brot,\n            'target_domain':f'all 2**{size*brot} assignments',\n            'all_clean_scratch_restored':True,\n            'all_selection_bits_restored':True,\n            'zero_phase':'Only unconditional X/CX/CCX permutation gates'}\n\n\ndef native_check(circuit, delta, brot, verifier, repeats=2):\n    import kickmix as km\n    path = Path(verifier)/'scoreinfra/circuits.py'\n    spec = importlib.util.spec_from_file_location('_organizer_lookup_contract',path)\n    module = importlib.util.module_from_spec(spec)\n    sys.modules[spec.name] = module\n    spec.loader.exec_module(module)\n    layout = module.register_layout([circuit])\n    count = len(delta)\n    addresses = np.tile(np.arange(count,dtype=np.int64),repeats)\n    rng = np.random.default_rng(902113)\n    initial = rng.integers(0,1 << brot,size=(len(addresses),delta.shape[1]))\n    bits = ((addresses[None,:] >> np.arange(len(layout.selection))[:,None]) & 1).astype(bool)\n    sim = km.Simulator(batch_size=len(addresses),debug_print=False,seed=902113,\n                       counting_operations=True)\n    sim.use_same_registers_as(circuit)\n    sim.write_across_shots(layout.selection,bits)\n    for j,reg in enumerate(layout.words):\n        sim.write_across_shots(reg,((initial[:,j][None,:] >> np.arange(brot)[:,None]) & 1).astype(bool))\n    sim.do(circuit)\n    module._check_state(sim,circuit,layout,bits,'organizer borrowed-prefix probe')\n    powers = np.int64(1) << np.arange(brot)\n    actual = np.stack([powers @ sim.read_across_shots(reg).astype(np.int64)\n                       for reg in layout.words],axis=1)\n    assert np.array_equal(actual, initial ^ delta[addresses])\n    assert sim.operation_statistics().num_toffoli_like == len(addresses)*circuit.operation_statistics().num_toffoli_like\n    return {'status':'PASS','actual_native_arbitrary_target_shots':len(addresses),\n            'scratch_phase_selection_checked':True}\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument('--angle-words',type=Path,required=True)\n    p.add_argument('--source-factorization',type=Path,required=True)\n    p.add_argument('--brot',type=int,default=12)\n    p.add_argument('--batch-size',type=int,default=9)\n    p.add_argument('--k',type=int,default=0)\n    p.add_argument('--clean-cap',type=int,default=8)\n    p.add_argument('--out',type=Path,required=True)\n    p.add_argument('--verifier-dir',type=Path,default=Path('/tmp/chem_verifiers/dfthcsos'))\n    a = p.parse_args()\n    start = time.time()\n    if a.out.exists():\n        raise FileExistsError('refusing to overwrite a probe directory')\n    words = np.load(a.angle_words,allow_pickle=False)\n    if isinstance(words,np.lib.npyio.NpzFile):\n        words = words['angle_words'].copy()\n    if np.any(words < 0) or np.any(words >= 1 << a.brot):\n        raise ValueError('word outside the selected precision')\n    delta = delta_batch(words,a.batch_size,a.k)\n    circuit, trace, report = build(delta,a.brot,a.clean_cap)\n    report['symbolic_oracle'] = symbolic_check(trace,delta,a.brot,circuit.num_qubits)\n    report['native_oracle'] = native_check(circuit,delta,a.brot,a.verifier_dir)\n    a.out.mkdir(parents=True)\n    cp = a.out/f'angle_k_{a.k}.kmb'\n    circuit.to_file(cp,format='kmb')\n    (a.out/f'angle_k_{a.k}.kmx').write_text(str(circuit))\n    np.save(a.out/'batch_delta.npy',delta)\n    report.update({'scope':'ORGANIZER_ONLY_EXACT_CIRCUIT_COMPONENT',\n        'full_family_compiled':False,'chemistry_acceptance_claimed':False,\n        'brot':a.brot,'batch_size':a.batch_size,'k':a.k,\n        'source_factorization':str(a.source_factorization.resolve()),\n        'source_factorization_sha256':sha(a.source_factorization),\n        'angle_words':str(a.angle_words.resolve()),'angle_words_sha256':sha(a.angle_words),\n        'compiler_sha256':sha(__file__),'native_sha256':sha(cp),\n        'seconds':time.time()-start})\n    (a.out/'report.json').write_text(json.dumps(report,indent=2)+'\\n')\n    print(json.dumps(report,indent=2))\n\n\nif __name__ == '__main__':\n    main()\n",
  "emit_j5.py": "# Derived from Ahmed Berat Ozer v8 emit_family.py (Apache 2.0).\n# New experiment: use five 8-bit outputs in freed QROM workspace;\n# select five relations per low-address block so the paired dimension is even.\n\"\"\"Derived J=5 / 8-bit ANGLE family for an arbitrary row count with mrmeows' exact paired-product lookup\n(Apache-2.0, source/chemistry_6p69/reproduction), generalized: the number of kernel relations per three-vertex\nblock is a search parameter, so tables with fewer high-address rows (fewer vectors) use their larger kernel.\n\nUsage: python emit_family.py <words .npy or submission .npz> <out dir> [jobs]\nWrites angle_k_*.kmb and plans.json; every circuit is natively checked on all addresses (64 targets each).\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport random\nimport sys\nimport time\nfrom concurrent.futures import ProcessPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\n\nREP=Path(__file__).resolve().parent\nsys.path.insert(0,str(REP))\nimport rank_parity as core  # noqa: E402\nimport kernel_planes as binary  # noqa: E402\nfrom permuted_kernel import quotient_selection  # noqa: E402\n\n\ndef plan_general(delta, rel_base=3, extra=15, seed=0, permutation=None):\n    permutation = list(range(15)) if permutation is None else list(permutation)\n    original_ac = core.coefficients(delta, 8, 4)\n    ac = original_ac.copy()\n    ac[:, 1:, :] = ac[:, 1:, :][:, permutation, :]\n    tensor = ac[1:, 1:, :]\n    H, L, m = tensor.shape\n    assert L == 15\n    rows = [binary.packed(row) for row in tensor.reshape(H, L * m)]\n    rng = random.Random(seed)\n    relations = []\n    chosen_extra = set(rng.sample(range(L), min(extra, L)))\n    for first in range(L):\n        vs = [(first + i) % L for i in range(3)]\n        block = np.concatenate([tensor[:, v, :] for v in vs], axis=1)\n        want = rel_base + (first in chosen_extra)\n        kern = core.nullspace([binary.packed(row) for row in block], 3 * m)\n        want = min(want, len(kern))\n        selected, _ = binary.select_relations([binary.packed(row) for row in block], 3 * m, want, rng)\n        for rel in selected:\n            relations.append(sum(((rel >> (j * m)) & ((1 << m) - 1)) << (v * m) for j, v in enumerate(vs)))\n    rb = {}\n    for r in relations:\n        binary.append_row(r, rb)\n    chunks = [[(r >> (j * m)) & ((1 << m) - 1) for j in range(L)] for r in relations]\n    planes = [(1 << i, 1 << j) for i in range(L) for j in range(i + 1, L)]\n    for step in [1, 2]:\n        planes.extend(((1 << i) | (1 << ((i + step) % L)), 1 << k) for i in range(L) for k in range(L)\n                      if k not in [i, (i + step) % L])\n    templates = []\n    for a, b in planes:\n        eq = []\n        for cs in chunks:\n            for ell in [a, b]:\n                value = 0\n                for j in core.set_bits(ell):\n                    value ^= cs[j]\n                if value:\n                    eq.append(value)\n        V = core.nullspace(eq, m)\n        directions = set(V)\n        if len(V) <= 6:\n            for code in range(1, 1 << len(V)):\n                direction = 0\n                for j in core.set_bits(code):\n                    direction ^= V[j]\n                directions.add(direction)\n        for v in sorted(directions):\n            if v:\n                templates.append((binary.tensor_column(v, a, m), binary.tensor_column(v, b, m), v, a, b))\n    span = {}\n    for a, b, *_ in templates:\n        binary.append_row(a, span)\n        binary.append_row(b, span)\n    target = L * m - len(rb)\n    contains = all(not binary.reduce_row(row, span) for row in rows)\n    if len(span) != target or not contains or target % 2:\n        return None, {'target': target, 'span': len(span), 'contains': contains}\n    selected, _ = quotient_selection(templates, target, L, seed + 7001)\n    if selected is None:\n        return None, {'target': target, 'paired': None}\n    basis = core.BinaryBasis()\n    for i in selected:\n        a, b, *_ = templates[i]\n        assert basis.add(a) and basis.add(b)\n    solutions = [basis.solve(row) for row in rows]\n    assert all(s is not None for s in solutions)\n    terms = []\n    reconstructed = [0] * H\n    for j, i in enumerate(selected):\n        a, b, v, ell1, ell2 = templates[i]\n        h1 = sum(((s >> (2 * j)) & 1) << h for h, s in enumerate(solutions))\n        h2 = sum(((s >> (2 * j + 1)) & 1) << h for h, s in enumerate(solutions))\n        terms.append({'target_direction': v, 'factors': [(h1, ell1), (h2, ell2)]})\n        for h in core.set_bits(h1):\n            reconstructed[h] ^= a\n        for h in core.set_bits(h2):\n            reconstructed[h] ^= b\n    assert reconstructed == rows\n    for term in terms:\n        term['factors'] = [(h, sum(1 << permutation[j] for j in core.set_bits(l))) for h, l in term['factors']]\n    return (original_ac, terms), {'target': target, 'paired': len(selected), 'relations': len(rb),\n                                  'rel_base': rel_base, 'extra': extra, 'seed': seed}\n\n\ndef best_plan(delta, budget_seconds=600, rel_bases=(4,)):\n    \"\"\"mrmeows' accepted plans all use 3 relations per block plus one extra on every block (60 in all),\n    with a searched seed and low-monomial permutation; search the same space until a plan exists.\"\"\"\n    t0 = time.time()\n    rng = random.Random(12345)\n    attempt = 0\n    while time.time() - t0 < budget_seconds:\n        perm = list(range(15))\n        if attempt:\n            rng.shuffle(perm)\n        for rel_base in rel_bases:\n            for seed in range(8):\n                try:\n                    res, rep = plan_general(delta, rel_base, 15, seed, perm)\n                except (ValueError, AssertionError):\n                    continue\n                if res is not None:\n                    rep['permutation'] = perm\n                    return res, rep\n        attempt += 1\n    return None\n\n\ndef build_one(args):\n    k, words, out = args\n    import kickmix as km\n    from hyper_native import build_hmr\n    from native_checks import check\n    delta = core.delta_batch(words, 5, k)\n    res = best_plan(delta)\n    if res is None:\n        return {'k': k, 'error': 'no plan'}\n    (ac, terms), rep = res\n    circuit, _, resources = build_hmr(delta, 8, 4, ac, terms)\n    path = Path(out) / f'angle_k_{k}.kmb'\n    circuit.to_file(path, format='kmb')\n    circuit = km.Circuit.from_file(path, format='kmb')\n    native = check(circuit, delta, brot=8, repeats=64, seed=150007 + k)\n    assert native == resources['toffoli_count']\n    return {'k': k, 'T': native, 'Q': int(circuit.num_qubits), **rep}\n\n\ndef main():\n    src, out = Path(sys.argv[1]), Path(sys.argv[2])\n    jobs = int(sys.argv[3]) if len(sys.argv) > 3 else 4\n    words = np.load(src)['angle_words'] if src.suffix == '.npz' else np.load(src)\n    words = np.asarray(words, dtype=np.int64)\n    out.mkdir(parents=True, exist_ok=True)\n    K = (words.shape[1] + 4) // 5\n    with ProcessPoolExecutor(max_workers=jobs) as pool:\n        reports = list(pool.map(build_one, [(k, words, str(out)) for k in range(K)]))\n    total = sum(r.get('T', 0) for r in reports)\n    (out / 'plans.json').write_text(json.dumps({'rows': int(words.shape[0]), 'per_pass': total, 'T_both_passes': 2 * total,\n                                                'circuits': reports}, indent=1))\n    print(json.dumps({'rows': int(words.shape[0]), 'per_pass': total, 'T_both': 2 * total,\n                      'per_circuit': [r.get('T') for r in reports], 'paired': [r.get('paired') for r in reports]}))\n\n\nif __name__ == '__main__':\n    main()\n",
  "extraction.json": "{\n  \"notebook\": \"/private/tmp/claude-501/-Users-flexonafft-MLKaggleTasks/3c4e21da-4677-4c8d-b41b-3cc340fff35a/scratchpad/w07/nbs/lightningv08-bezout-schur-5-word-qrom/bezout-schur-5-word-qrom.ipynb\",\n  \"cell\": 2,\n  \"method\": \"ast.literal_eval of COMPILER_SOURCE JSON; no code executed\",\n  \"sha256\": \"17a0cc7b03d283fb2f7b2816fb4aff85090b1c43c602812a7890e0459bab399f\"\n}",
  "hyper_native.py": "from pathlib import Path\nimport sys\nfrom rank_parity import *\ndef build_hmr(delta,brot,lowbits,ac,terms):\n    import kickmix as km\n    delta = require_redundant_sign_bit(delta,brot)\n    D,J = delta.shape\n    n = (D-1).bit_length()\n    A,B,O = ac.shape\n    H,L = A-1,B-1\n    builder = km.CircuitBuilder()\n    qubits = list(builder.create_quantum_register(n,'selection'))\n    for j in range(J):\n        qubits += list(builder.create_quantum_register(brot,f'word_{j}'))\n    trace = []\n    def gate(name,*ids):\n        trace.append((name,*ids))\n        getattr(builder,name.lower())(*[qubits[i] for i in ids])\n    cache = []\n    def cached_basis(size,offset):\n        result = {0:None}\n        for mask in range(1,size):\n            bit = mask.bit_length()-1\n            if mask.bit_count()==1:\n                result[mask] = offset+bit\n            else:\n                parent = mask ^ (1 << bit)\n                q = len(qubits)\n                qubits.extend(builder.alloc_qubits(1))\n                result[mask] = q\n                triple = (result[parent],offset+bit,q)\n                gate('CCX',*triple)\n                cache.append(triple)\n        return result\n    high = cached_basis(A,lowbits)\n    low = cached_basis(B,0)\n    if len(qubits)>128:\n        raise ValueError(f'cache layout needs {len(qubits)} qubits')\n    basis = [high[i] for i in range(1,A)]+[low[i] for i in range(1,B)]\n    groups = [[n+j*brot+bit] + ([n+j*brot+brot-1] if bit==brot-2 else [])\n              for j in range(J) for bit in range(brot-1)]\n    def target_bits(direction):\n        return [q for j in set_bits(direction) for q in groups[j]]\n    def linear_output(control,direction):\n        for q in target_bits(direction):\n            gate('CX',control,q)\n    def product(a,b,direction):\n        if not a or not b or not direction:\n            return\n        if a == b:\n            for i in set_bits(a):\n                linear_output(basis[i],direction)\n            return\n        p = (a & -a).bit_length()-1\n        first = list(set_bits(a ^ (1 << p)))\n        for i in first:\n            gate('CX',basis[i],basis[p])\n        if b >> p & 1:\n            b ^= a ^ (1 << p)\n        choices = b & ~(1 << p)\n        assert choices\n        q = (choices & -choices).bit_length()-1\n        second = list(set_bits(b ^ (1 << q)))\n        for i in second:\n            gate('CX',basis[i],basis[q])\n        pivot,*others = target_bits(direction)\n        for out in others:\n            gate('CX',pivot,out)\n        gate('CCX',basis[p],basis[q],pivot)\n        for out in others:\n            gate('CX',pivot,out)\n        for i in reversed(second):\n            gate('CX',basis[i],basis[q])\n        for i in reversed(first):\n            gate('CX',basis[i],basis[p])\n    high_only = [sum(int(ac[h,0,j]) << j for j in range(O)) for h in range(A)]\n    low_only = [sum(int(ac[0,l,j]) << j for j in range(O)) for l in range(B)]\n    low_only[0] = 0\n    def correction(destination,left,right,direction):\n        for a in set_bits(left):\n            for b in set_bits(right):\n                index = (a+1)|(b+1)\n                if index < len(destination):\n                    destination[index] ^= direction\n    for term in terms:\n        pairs = term['factors']\n        direction = term['target_direction']\n        for i in range(0,len(pairs),2):\n            h1,l1 = pairs[i]\n            if i+1 == len(pairs):\n                product(h1,l1 << H,direction)\n            else:\n                h2,l2 = pairs[i+1]\n                # (h1+l2)(h2+l1) + h1*h2 + l1*l2\n                # equals h1*l1 + h2*l2 over GF(2).\n                product(h1 ^ (l2 << H),h2 ^ (l1 << H),direction)\n                correction(high_only,h1,h2,direction)\n                correction(low_only,l1,l2,direction)\n    for q in target_bits(high_only[0]):\n        gate('X',q)\n    for mask in range(1,A):\n        linear_output(high[mask],high_only[mask])\n    for mask in range(1,B):\n        linear_output(low[mask],low_only[mask])\n    for a,b,q in reversed(cache):\n        builder.del_and(qubits[a],qubits[b],qubits[q])\n    circuit = builder.finish_circuit()\n    native = int(circuit.operation_statistics().num_toffoli_like)\n    assert native == sum(t[0]=='CCX' for t in trace)\n    return circuit,trace,{'num_qubits':int(circuit.num_qubits),\n        'toffoli_count':native,'clean_cached_functions':len(cache),\n        'cache_compute_toffolis':len(cache), 'cache_HMR_cleanup':len(cache),\n        'gates':len(trace)}\n\n",
  "kernel_planes.py": "#!/usr/bin/env python3\n\"\"\"Exact multi-output product sharing on a single organizer-only fixture.\n\nThe search chooses a small subspace of the coefficient tensor's exact kernel.\nRank-two product templates are admitted only when BOTH of their coefficient\ncolumns annihilate those relations. A greedy paired basis search is accepted\nonly after exact reconstruction of every tensor coefficient. No randomized\nrank test or chemistry approximation is used as an acceptance criterion.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nimport random\nimport time\n\nimport numpy as np\n\nimport rank_parity as core\n\n\ndef packed(row):\n    return sum(int(value) << j for j,value in enumerate(row))\n\n\ndef tensor_column(direction,lowform,m):\n    return sum(direction << (m*j) for j in core.set_bits(lowform))\n\n\ndef low_planes():\n    planes = set()\n    def add(a,b):\n        first,second,_ = sorted((a,b,a^b))\n        planes.add((first,second))\n    for shift in (0,3):\n        for a in range(1,8):\n            for b in range(a+1,8):\n                add(a << shift,b << shift)\n    for a in range(1,8):\n        for b in range(1,8):\n            add(a,b << 3)\n        add(a,1 << 6)\n        add(a << 3,1 << 6)\n    return sorted(planes)\n\n\ndef reduce_row(value,pivots):\n    while value:\n        p = value.bit_length()-1\n        if p not in pivots:\n            break\n        value ^= pivots[p]\n    return value\n\n\ndef append_row(value,pivots):\n    value = reduce_row(value,pivots)\n    if not value:\n        return False\n    pivots[value.bit_length()-1] = value\n    return True\n\n\ndef select_relations(rows,width,dimension,rng):\n    kernel = core.nullspace(rows,width)\n    if len(kernel) < dimension:\n        raise ValueError('requested kernel dimension exceeds the actual nullity')\n    chosen = []\n    span = {}\n    while len(chosen) < dimension:\n        mask = rng.getrandbits(len(kernel))\n        value = 0\n        for j in core.set_bits(mask):\n            value ^= kernel[j]\n        if append_row(value,span):\n            assert all((value & row).bit_count()%2 == 0 for row in rows)\n            chosen.append(value)\n    return chosen,len(kernel)\n\n\ndef paired_basis(templates,target_rank,seed,trials):\n    rng = random.Random(seed)\n    best = 0\n    reports = []\n    for trial in range(trials):\n        order = list(range(len(templates)))\n        rng.shuffle(order)\n        span = {}\n        chosen = []\n        for i in order:\n            a,b,*_ = templates[i]\n            ar = reduce_row(a,span)\n            if not ar:\n                continue\n            p = ar.bit_length()-1\n            span[p] = ar\n            br = reduce_row(b,span)\n            if not br:\n                del span[p]\n                continue\n            span[br.bit_length()-1] = br\n            chosen.append(i)\n            if len(span) == target_rank:\n                reports.append({'trial':trial,'rank':len(span)})\n                return chosen,reports\n        best = max(best,len(span))\n        reports.append({'trial':trial,'rank':len(span)})\n    return None,reports\n\n\ndef plan(delta,brot,dimensions=(15,15),seed=0,pair_trials=16,samples=12):\n    core.require_redundant_sign_bit(delta,brot)\n    ac = core.coefficients(delta,brot,3)\n    tensor = ac[1:,1:,:]\n    H,L,m = tensor.shape\n    if L != 7:\n        raise ValueError('this bounded planner requires exactly three low address bits')\n    F = tensor.reshape(H,L*m)\n    rows = [packed(row) for row in F]\n    rng = random.Random(seed)\n    relations = []\n    nullities = []\n    for group,dimension in enumerate(dimensions):\n        block_rows = [packed(row) for row in F[:,3*m*group:3*m*(group+1)]]\n        chosen,nullity = select_relations(block_rows,3*m,dimension,rng)\n        relations += [value << (3*m*group) for value in chosen]\n        nullities.append(nullity)\n    assert all((row & relation).bit_count()%2 == 0 for row in rows for relation in relations)\n    mask = (1 << m)-1\n    chunks = [[(r >> (j*m)) & mask for j in range(L)] for r in relations]\n    templates = []\n    plane_dimensions = []\n    for ell1,ell2 in low_planes():\n        equations = []\n        for row in chunks:\n            for ell in (ell1,ell2):\n                value = 0\n                for j in core.set_bits(ell):\n                    value ^= row[j]\n                if value:\n                    equations.append(value)\n        V = core.nullspace(equations,m)\n        plane_dimensions.append(len(V))\n        directions = set(V)\n        for _ in range(samples):\n            choice = rng.getrandbits(len(V))\n            value = 0\n            for j in core.set_bits(choice):\n                value ^= V[j]\n            if value:\n                directions.add(value)\n        for direction in sorted(directions):\n            a = tensor_column(direction,ell1,m)\n            b = tensor_column(direction,ell2,m)\n            assert all((a&r).bit_count()%2 == 0 and (b&r).bit_count()%2 == 0 for r in relations)\n            templates.append((a,b,direction,ell1,ell2))\n    all_span = {}\n    for a,b,*_ in templates:\n        append_row(a,all_span)\n        append_row(b,all_span)\n    target_rank = L*m-sum(dimensions)\n    contains_target = all(not reduce_row(row,all_span) for row in rows)\n    report = {'H':H,'L':L,'O':m,'kernel_dimensions':list(dimensions),\n        'kernel_nullities':nullities,'kernel_relations':relations,\n        'num_templates':len(templates),'template_span_rank':len(all_span),\n        'desired_span_rank':target_rank,'contains_target':contains_target,\n        'plane_output_nullities':plane_dimensions,'seed':seed}\n    if len(all_span) != target_rank or not contains_target:\n        return None,report\n    indices,attempts = paired_basis(templates,target_rank,seed+1001,pair_trials)\n    report['paired_basis_attempts'] = attempts\n    if indices is None:\n        return None,report\n    chosen = [templates[i] for i in indices]\n    basis = core.BinaryBasis()\n    for a,b,*_ in chosen:\n        assert basis.add(a) and basis.add(b)\n    solutions = [basis.solve(row) for row in rows]\n    assert all(value is not None for value in solutions)\n    terms = []\n    reconstructed = [0]*H\n    for i,(a,b,direction,ell1,ell2) in enumerate(chosen):\n        h1 = sum(((value >> (2*i))&1) << h for h,value in enumerate(solutions))\n        h2 = sum(((value >> (2*i+1))&1) << h for h,value in enumerate(solutions))\n        terms.append({'target_direction':direction,'factors':[(h1,ell1),(h2,ell2)]})\n        for h in core.set_bits(h1):\n            reconstructed[h] ^= a\n        for h in core.set_bits(h2):\n            reconstructed[h] ^= b\n    assert reconstructed == rows\n    report.update({'selected_pairs':len(chosen),'exact_selected_binary_rank':len(basis.vectors),\n        'exact_tensor_reconstruction':True,'selected_template_indices':indices})\n    return (ac,terms),report\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument('--fixture-dir',type=Path,required=True)\n    p.add_argument('--dimensions',type=int,nargs=2,default=(15,15))\n    p.add_argument('--seed',type=int,default=0)\n    p.add_argument('--pair-trials',type=int,default=16)\n    p.add_argument('--samples',type=int,default=12)\n    p.add_argument('--out',type=Path,required=True)\n    p.add_argument('--verifier-dir',type=Path,default=Path('/tmp/chem_verifiers/dfthcsos'))\n    a = p.parse_args()\n    start = time.time()\n    if a.out.exists():\n        raise FileExistsError('refusing to overwrite an experiment')\n    source = json.loads((a.fixture_dir/'report.json').read_text())\n    if source['scope'] != 'ORGANIZER_SUBSET_CIRCUIT_FIXTURE_ONLY':\n        raise ValueError('this bounded experiment requires the organizer fixture')\n    delta = np.load(a.fixture_dir/'batch_delta.npy',allow_pickle=False)\n    result,planning = plan(delta,source['brot'],tuple(a.dimensions),a.seed,a.pair_trials,a.samples)\n    a.out.mkdir(parents=True)\n    if result is None:\n        report={'status':'NO_EXACT_PAIRED_PLAN','planning':planning,'seconds':time.time()-start}\n        (a.out/'report.json').write_text(json.dumps(report,indent=2)+'\\n')\n        print(json.dumps({k:v for k,v in report.items() if k!='planning'}))\n        print(json.dumps({k:v for k,v in planning.items() if k not in ['kernel_relations','plane_output_nullities']}))\n        return\n    ac,terms = result\n    circuit,trace,report = core.build(delta,source['brot'],3,ac,terms)\n    report['exact_symbolic_oracle'] = core.symbolic_check(trace,delta,source['brot'],circuit.num_qubits)\n    report['native_oracle'] = core.native_check(circuit,delta,source['brot'],a.verifier_dir)\n    cp = a.out/'angle_k_0.kmb'\n    circuit.to_file(cp,format='kmb')\n    (a.out/'angle_k_0.kmx').write_text(str(circuit))\n    (a.out/'certificate.json').write_text(json.dumps({'planning':planning,'terms':terms},indent=2)+'\\n')\n    report.update({'status':'PASS','scope':'ORGANIZER_SUBSET_CIRCUIT_FIXTURE_ONLY',\n        'chemistry_acceptance_claimed':False,'full_family_compiled':False,\n        'D':len(delta),'brot':source['brot'],'batch_size':source['batch_size'],\n        'planning':planning,'fixture_source_report_sha256':core.sha(a.fixture_dir/'report.json'),\n        'fixture_delta_sha256':core.sha(a.fixture_dir/'batch_delta.npy'),\n        'planner_sha256':core.sha(__file__),'emitter_sha256':core.sha(core.__file__),\n        'proof_helper_sha256':core.sha(Path(__file__).with_name('borrowed_prefix.py')),\n        'native_sha256':core.sha(cp),'seconds':time.time()-start})\n    (a.out/'report.json').write_text(json.dumps(report,indent=2)+'\\n')\n    print(json.dumps({k:v for k,v in report.items() if k!='planning'},indent=2))\n    print(json.dumps({k:v for k,v in planning.items() if k not in ['kernel_relations','plane_output_nullities']},indent=2))\n\n\nif __name__=='__main__':\n    main()\n",
  "native_checks.py": "import numpy as np\nimport kickmix as km\n\ndef check(circuit, delta, *, brot=11, repeats=4, seed=127813):\n    \"\"\"Check every valid address with arbitrary targets, phase and scratch.\"\"\"\n    count, word_count = delta.shape\n    n_sel = (count - 1).bit_length()\n    addresses = np.tile(np.arange(count), repeats)\n    rng = np.random.default_rng(seed)\n    initial = rng.integers(0, 1 << brot, size=(len(addresses), word_count))\n    if repeats >= 2:\n        initial[:count] = 0\n        initial[count:2*count] = (1 << brot) - 1\n    sim = km.Simulator(batch_size=len(addresses), seed=seed, counting_operations=True)\n    sim.use_same_registers_as(circuit)\n    sim.write_across_shots(\"selection\", ((addresses[None, :] >> np.arange(n_sel)[:, None]) & 1).astype(bool))\n    for j in range(word_count):\n        sim.write_across_shots(f\"word_{j}\", ((initial[:, j][None, :] >> np.arange(brot)[:, None]) & 1).astype(bool))\n    powers = np.int64(1) << np.arange(brot)\n    for application in range(2):\n        sim.do(circuit)\n        got = np.stack([powers @ sim.read_across_shots(f\"word_{j}\").astype(np.int64)\n                        for j in range(word_count)], axis=1)\n        expected = initial ^ delta[addresses] if application == 0 else initial\n        assert np.array_equal(got, expected), \"incorrect arbitrary-target XOR\"\n        assert not np.any(sim.read_phase_flipped_across_shots()), \"incorrect phase\"\n        decoded = (np.int64(1) << np.arange(n_sel)) @ sim.read_across_shots(\"selection\").astype(np.int64)\n        assert np.array_equal(decoded, addresses), \"selection changed\"\n        total = sim.read_across_shots(km.array([km.q(i) for i in range(circuit.num_qubits)]))\n        assert not np.any(total[n_sel + brot * word_count:]), \"scratch not returned to zero\"\n    return float(sim.operation_statistics().num_toffoli_like) / len(addresses) / 2\n",
  "permuted_kernel.py": "\"\"\"Three-low-direction kernel cycles for exact J4 bilinear synthesis.\"\"\"\nfrom pathlib import Path\nimport sys,json,time,random,itertools\nimport numpy as np\nROOT=Path(__file__).resolve().parent\nsys.path.insert(0,str(ROOT))\nimport rank_parity as core\nimport kernel_planes as binary\n\ndef quotient_selection(templates,target,L,seed,trials=64):\n    W0={};coordinate=[];spaces=[{} for _ in range(L)]\n    for i,(a,b,v,ell1,ell2) in enumerate(templates):\n        if ell1.bit_count()==ell2.bit_count()==1:\n            left=ell1.bit_length()-1;right=ell2.bit_length()-1\n            coordinate.append((i,left,right,v))\n            binary.append_row(a,W0);binary.append_row(b,W0)\n            binary.append_row(v,spaces[left]);binary.append_row(v,spaces[right])\n    pivots=sorted(W0.items(),reverse=True)\n    def quotient(v):\n        for p,row in pivots:\n            if v>>p&1:v^=row\n        return v\n    mixed=[]\n    for i,(a,b,v,ell1,ell2) in enumerate(templates):\n        ar,br=quotient(a),quotient(b)\n        if ar and not br and ell2.bit_count()==1:mixed.append((i,ar,ell2.bit_length()-1,v))\n        elif br and not ar and ell1.bit_count()==1:mixed.append((i,br,ell1.bit_length()-1,v))\n    qrank=target-len(W0);dims=[len(x) for x in spaces];rng=random.Random(seed);attempts=[]\n    for trial in range(trials):\n        order=mixed.copy();rng.shuffle(order);qspan={};local=[{} for _ in range(L)];chosen=[]\n        for i,q,vertex,v in order:\n            qr=binary.reduce_row(q,qspan);vr=binary.reduce_row(v,local[vertex])\n            if qr and vr:\n                qspan[qr.bit_length()-1]=qr;local[vertex][vr.bit_length()-1]=vr;chosen.append(i)\n        if len(qspan)!=qrank:\n            attempts.append({'trial':trial,'quotient_rank':len(qspan),'required':qrank});continue\n        pool=coordinate.copy();rng.shuffle(pool)\n        while 2*len(chosen)<target:\n            deficit=[dims[j]-len(local[j]) for j in range(L)];best=None\n            for i,left,right,v in pool:\n                if not deficit[left] or not deficit[right]:continue\n                lr=binary.reduce_row(v,local[left]);rr=binary.reduce_row(v,local[right])\n                if not lr or not rr:continue\n                score=(deficit[left]+deficit[right],min(deficit[left],deficit[right]))\n                if best is None or score>best[0]:best=(score,i,left,right,lr,rr)\n            if best is None:break\n            _,i,left,right,lr,rr=best;local[left][lr.bit_length()-1]=lr;local[right][rr.bit_length()-1]=rr;chosen.append(i)\n        span={}\n        for i in chosen:\n            binary.append_row(templates[i][0],span);binary.append_row(templates[i][1],span)\n        attempts.append({'trial':trial,'quotient_rank':len(qspan),'rank':len(span),'pairs':len(chosen)})\n        if 2*len(chosen)==len(span)==target:return chosen,{'W0':len(W0),'qrank':qrank,'attempts':attempts}\n    return None,{'W0':len(W0),'qrank':qrank,'attempts':attempts}\n\ndef plan(delta,extra=15,seed=0,permutation=None):\n    permutation=list(range(15)) if permutation is None else list(permutation)\n    assert sorted(permutation)==list(range(15))\n    original_ac=core.coefficients(delta,8,4); ac=original_ac.copy(); ac[:,1:,:]=ac[:,1:,:][:,permutation,:]; tensor=ac[1:,1:,:];H,L,m=tensor.shape\n    assert (L,m)==(15,28)\n    rows=[binary.packed(row) for row in tensor.reshape(H,L*m)]\n    rng=random.Random(seed);relations=[];nullities=[]\n    chosen_extra=set(rng.sample(range(L),extra))\n    for first in range(L):\n        vs=[(first+i)%L for i in range(3)]\n        block=np.concatenate([tensor[:,v,:] for v in vs],axis=1)\n        selected,nullity=binary.select_relations([binary.packed(row) for row in block],3*m,3+(first in chosen_extra),rng)\n        nullities.append(nullity)\n        for rel in selected:\n            relations.append(sum(((rel>>(j*m))&((1<<m)-1))<<(v*m) for j,v in enumerate(vs)))\n    rb={}\n    for r in relations:binary.append_row(r,rb)\n    assert all(not ((row&r).bit_count()&1) for row in rows for r in relations)\n    chunks=[[(r>>(j*m))&((1<<m)-1) for j in range(L)] for r in relations]\n    planes=[(1<<i,1<<j) for i in range(L) for j in range(i+1,L)]\n    for step in [1,2]:\n        planes.extend(((1<<i)|(1<<((i+step)%L)),1<<k) for i in range(L) for k in range(L) if k not in [i,(i+step)%L])\n    templates=[];dims=[]\n    for a,b in planes:\n        eq=[]\n        for cs in chunks:\n            for ell in [a,b]:\n                value=0\n                for j in core.set_bits(ell):value^=cs[j]\n                if value:eq.append(value)\n        V=core.nullspace(eq,m);dims.append(len(V))\n        directions=set(V)\n        if len(V)<=6:\n            for code in range(1,1<<len(V)):\n                direction=0\n                for j in core.set_bits(code):direction^=V[j]\n                directions.add(direction)\n        for v in sorted(directions):\n            if v:\n                u=binary.tensor_column(v,a,m);w=binary.tensor_column(v,b,m)\n                templates.append((u,w,v,a,b))\n    span={}\n    for a,b,*_ in templates:binary.append_row(a,span);binary.append_row(b,span)\n    target=L*m-len(rb)\n    report={'extra':extra,'seed':seed,'nullities':nullities,'relations':len(rb),'target_rank':target,'span_rank':len(span),'templates':len(templates),'plane_dim_hist':{i:dims.count(i) for i in sorted(set(dims))},'contains_target':all(not binary.reduce_row(row,span) for row in rows),'T_lower_if_paired':84+(target+1)//2}\n    if len(span)!=target or not report['contains_target'] or target%2:return None,report\n    selected,attempts=quotient_selection(templates,target,L,seed+7001)\n    report['paired_attempts']=attempts\n    if selected is None:return None,report\n    basis=core.BinaryBasis()\n    for i in selected:\n        a,b,*_=templates[i];assert basis.add(a) and basis.add(b)\n    solutions=[basis.solve(row) for row in rows];assert all(s is not None for s in solutions)\n    terms=[];reconstructed=[0]*H\n    for j,i in enumerate(selected):\n        a,b,v,ell1,ell2=templates[i]\n        h1=sum(((s>>(2*j))&1)<<h for h,s in enumerate(solutions));h2=sum(((s>>(2*j+1))&1)<<h for h,s in enumerate(solutions))\n        terms.append({'target_direction':v,'factors':[(h1,ell1),(h2,ell2)]})\n        for h in core.set_bits(h1):reconstructed[h]^=a\n        for h in core.set_bits(h2):reconstructed[h]^=b\n    assert reconstructed==rows\n    report.update({'paired_products':len(selected),'T_HMR':84+len(selected),'Q':127,'exact_tensor':True})\n    \n    for term in terms:\n        term['factors']=[(h,sum(1<<permutation[j] for j in core.set_bits(l))) for h,l in term['factors']]\n    report['low_permutation']=permutation\n    return (original_ac,terms),report\n\n",
  "published-runtime-receipt.json": "{\n  \"runtime\": {\n    \"python\": \"3.13.15\",\n    \"numpy\": \"2.1.3\",\n    \"kickmix\": \"0.1\"\n  },\n  \"salt\": \"lightningv08-v13-fixed-20261026\",\n  \"wall_seconds\": {\n    \"invsqrt\": 636.2630051360001,\n    \"dfthcsos\": 28.394002893999982,\n    \"dqi\": 426.736635654\n  },\n  \"warning\": \"different 1096-row table, not a benchmark of the proposed search\"\n}",
  "rank_parity.py": "#!/usr/bin/env python3\n\"\"\"Independent coherent lookup via an even-rank output-function basis.\n\nAll emitted gates are unconditional X/CX/CCX. No decomposition certificate or\ncandidate data is imported. The optional organizer subset is a circuit fixture,\nnot a chemistry submission. A rank-one matrix count is never reported as the\nnative paired-product count.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nimport numpy as np\n\nfrom borrowed_prefix import delta_batch, native_check, symbolic_check\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef set_bits(value):\n    while value:\n        bit = (value & -value).bit_length()-1\n        yield bit\n        value &= value-1\n\n\ndef mobius_axis(values, axis):\n    out = np.swapaxes(values.copy(),axis,0)\n    ids = np.arange(len(out))\n    for bit in range((len(out)-1).bit_length()):\n        rows = ids[(ids & (1 << bit)) != 0]\n        out[rows] ^= out[rows ^ (1 << bit)]\n    return np.swapaxes(out,0,axis)\n\n\ndef canonical_gauge(words, edges, brot):\n    \"\"\"Exact integer projective gauge; all cosines use a positive half-turn.\"\"\"\n    out = np.asarray(words,dtype=np.int64).copy()\n    signs = np.ones((len(words),len(edges)+1),dtype=np.int8)\n    modulus = 1 << brot\n    for k in reversed(range(len(edges))):\n        p,q = map(int,edges[k])\n        x = out[:,k].copy()\n        opposite = signs[:,p] != signs[:,q]\n        x[opposite] = (-x[opposite]) % modulus\n        flip = (x >= modulus//4) & (x < 3*modulus//4)\n        x[flip] = (x[flip]+modulus//2) % modulus\n        signs[flip,p] *= -1\n        signs[:,q] = 1\n        out[:,k] = x\n    assert not np.any(((out >> (brot-1)) ^ (out >> (brot-2))) & 1)\n    return out, signs[:,0]\n\n\ndef decode(words,edges,brot):\n    v = np.zeros((len(words),len(edges)+1))\n    v[:,0] = 1\n    for k,(p,q) in enumerate(edges):\n        theta = words[:,k]*(np.pi/(1 << (brot-1)))\n        left,right = v[:,p].copy(),v[:,q].copy()\n        v[:,p] = np.cos(theta)*left - np.sin(theta)*right\n        v[:,q] = np.sin(theta)*left + np.cos(theta)*right\n    return v\n\n\nclass BinaryBasis:\n    def __init__(self):\n        self.pivots = {}\n        self.vectors = []\n\n    def add(self,value):\n        original = value\n        coordinates = 1 << len(self.vectors)\n        while value:\n            p = value.bit_length()-1\n            if p not in self.pivots:\n                self.pivots[p] = (value,coordinates)\n                self.vectors.append(original)\n                return True\n            v,c = self.pivots[p]\n            value ^= v\n            coordinates ^= c\n        return False\n\n    def solve(self,value):\n        coordinates = 0\n        while value:\n            p = value.bit_length()-1\n            if p not in self.pivots:\n                return None\n            v,c = self.pivots[p]\n            value ^= v\n            coordinates ^= c\n        return coordinates\n\n\ndef nullspace(rows, width):\n    pivots = {}\n    for value in rows:\n        value = int(value)\n        while value:\n            p = value.bit_length()-1\n            if p not in pivots:\n                pivots[p] = value\n                if len(pivots) == width:\n                    return []\n                break\n            value ^= pivots[p]\n    result = []\n    for j in range(width):\n        if j not in pivots:\n            value = 1 << j\n            for p,row in sorted(pivots.items()):\n                if (row & value).bit_count() & 1:\n                    value ^= 1 << p\n            result.append(value)\n    return result\n\n\ndef factor_matrix(columns):\n    basis = BinaryBasis()\n    for col in columns:\n        basis.add(col)\n    lows = [0]*len(basis.vectors)\n    for k,col in enumerate(columns):\n        coordinates = basis.solve(col)\n        assert coordinates is not None\n        for bit in set_bits(coordinates):\n            lows[bit] ^= 1 << k\n    return list(zip(basis.vectors,lows))\n\n\ndef require_redundant_sign_bit(delta,brot):\n    \"\"\"Reject raw tables that cannot be represented by b-1 Boolean functions.\n\n    Every physical word bit remains an arbitrary target. This precondition is\n    about the requested XOR values, not an assumption about target contents.\n    The caller must explicitly apply a legal projective gauge beforehand.\n    \"\"\"\n    if not isinstance(brot,(int,np.integer)) or not 2 <= brot <= 62:\n        raise ValueError('brot must be an integer between 2 and 62')\n    table = np.asarray(delta)\n    if (table.ndim != 2 or min(table.shape,default=0) < 1 or\n            not np.issubdtype(table.dtype,np.integer)):\n        raise ValueError('delta must be a nonempty two-dimensional integer table')\n    if np.any(table < 0) or np.any(table >= (1 << brot)):\n        raise ValueError('delta word outside the selected precision')\n    table = table.astype(np.int64,copy=False)\n    if np.any(((table >> (brot-1)) ^ (table >> (brot-2))) & 1):\n        raise ValueError('redundant sign-bit precondition failed: the two highest '\n                         'bits of every requested XOR word must be equal')\n    return table\n\n\ndef coefficients(delta,brot,lowbits):\n    delta = require_redundant_sign_bit(delta,brot)\n    count,J = delta.shape\n    low = 1 << lowbits\n    high = (count+low-1)//low\n    padded = np.zeros((low*high,J),dtype=np.int64)\n    padded[:count] = delta\n    require_redundant_sign_bit(padded,brot)\n    bits = np.stack([(padded[:,j] >> bit)&1 for j in range(J)\n                     for bit in range(brot-1)],axis=1).astype(np.uint8)\n    truth = bits.reshape(high,low,-1)\n    result = mobius_axis(mobius_axis(truth,0),1)\n    assert np.array_equal(mobius_axis(mobius_axis(result,1),0),truth)\n    return result\n\n\ndef plan(delta,brot,lowbits,max_kernels=32767):\n    ac = coefficients(delta,brot,lowbits)\n    tensor = ac[1:,1:,:]\n    H,L,O = tensor.shape\n    columns = [[sum(int(tensor[h,k,j]) << h for h in range(H))\n                for k in range(L)] for j in range(O)]\n    original_rank = [len(factor_matrix(cols)) for cols in columns]\n    chosen = BinaryBasis()\n    certificates = []\n    # Preserve existing already-even or unusually-low rank directions first.\n    for j,r in sorted(enumerate(original_rank),key=lambda item:(item[1]+1)//2):\n        if r < min(H,L) or r % 2 == 0:\n            if chosen.add(1 << j):\n                certificates.append({'direction':1 << j,'kernel':None,'rank':r})\n    packed_low = [[sum(int(tensor[h,k,j]) << j for j in range(O))\n                   for h in range(H)] for k in range(L)]\n    current = [0]*H\n    previous = 0\n    searched = 0\n    for counter in range(1,min((1 << L)-1,max_kernels)+1):\n        mask = counter ^ (counter >> 1)\n        changed = (mask ^ previous).bit_length()-1\n        current = [a ^ b for a,b in zip(current,packed_low[changed])]\n        previous = mask\n        searched += 1\n        for direction in nullspace(current,O):\n            if chosen.solve(direction) is not None:\n                continue\n            mixture = [0]*L\n            for j in set_bits(direction):\n                mixture = [a ^ b for a,b in zip(mixture,columns[j])]\n            rank = len(factor_matrix(mixture))\n            # A rank drop must actually cross a paired-product threshold.\n            if (rank+1)//2 < (min(H,L)+1)//2:\n                assert chosen.add(direction)\n                certificates.append({'direction':direction,'kernel':mask,'rank':rank})\n        if len(chosen.vectors) == O:\n            break\n    for j in range(O):\n        if chosen.add(1 << j):\n            certificates.append({'direction':1 << j,'kernel':None,\n                                 'rank':original_rank[j]})\n    assert len(chosen.vectors) == O\n    output_directions = [0]*O\n    for j in range(O):\n        inverse_column = chosen.solve(1 << j)\n        assert inverse_column is not None\n        for i in set_bits(inverse_column):\n            output_directions[i] |= 1 << j\n    terms = []\n    reconstructed = [[0]*L for _ in range(O)]\n    products = 0\n    for i,vector in enumerate(chosen.vectors):\n        mixture = [0]*L\n        for j in set_bits(vector):\n            mixture = [a ^ b for a,b in zip(mixture,columns[j])]\n        factors = factor_matrix(mixture)\n        assert len(factors) == certificates[i]['rank']\n        terms.append({'input_output_basis':vector,\n                      'target_direction':output_directions[i],\n                      'factors':factors})\n        products += (len(factors)+1)//2\n        for j in set_bits(output_directions[i]):\n            reconstructed[j] = [a ^ b for a,b in zip(reconstructed[j],mixture)]\n    assert reconstructed == columns\n    return ac,terms,{'H':H,'L':L,'O':O,'searched_nonzero_low_kernels':searched,\n        'original_matrix_ranks':original_rank,'output_basis_certificates':certificates,\n        'untransformed_paired_products':sum((r+1)//2 for r in original_rank),\n        'transformed_paired_products':products,'exact_mixed_tensor_reconstruction':True}\n\n\ndef build(delta,brot,lowbits,ac,terms):\n    import kickmix as km\n    delta = require_redundant_sign_bit(delta,brot)\n    D,J = delta.shape\n    n = (D-1).bit_length()\n    A,B,O = ac.shape\n    H,L = A-1,B-1\n    builder = km.CircuitBuilder()\n    qubits = list(builder.create_quantum_register(n,'selection'))\n    for j in range(J):\n        qubits += list(builder.create_quantum_register(brot,f'word_{j}'))\n    trace = []\n    def gate(name,*ids):\n        trace.append((name,*ids))\n        getattr(builder,name.lower())(*[qubits[i] for i in ids])\n    cache = []\n    def cached_basis(size,offset):\n        result = {0:None}\n        for mask in range(1,size):\n            bit = mask.bit_length()-1\n            if mask.bit_count()==1:\n                result[mask] = offset+bit\n            else:\n                parent = mask ^ (1 << bit)\n                q = len(qubits)\n                qubits.extend(builder.alloc_qubits(1))\n                result[mask] = q\n                triple = (result[parent],offset+bit,q)\n                gate('CCX',*triple)\n                cache.append(triple)\n        return result\n    high = cached_basis(A,lowbits)\n    low = cached_basis(B,0)\n    if len(qubits)>128:\n        raise ValueError(f'cache layout needs {len(qubits)} qubits')\n    basis = [high[i] for i in range(1,A)]+[low[i] for i in range(1,B)]\n    groups = [[n+j*brot+bit] + ([n+j*brot+brot-1] if bit==brot-2 else [])\n              for j in range(J) for bit in range(brot-1)]\n    def target_bits(direction):\n        return [q for j in set_bits(direction) for q in groups[j]]\n    def linear_output(control,direction):\n        for q in target_bits(direction):\n            gate('CX',control,q)\n    def product(a,b,direction):\n        if not a or not b or not direction:\n            return\n        if a == b:\n            for i in set_bits(a):\n                linear_output(basis[i],direction)\n            return\n        p = (a & -a).bit_length()-1\n        first = list(set_bits(a ^ (1 << p)))\n        for i in first:\n            gate('CX',basis[i],basis[p])\n        if b >> p & 1:\n            b ^= a ^ (1 << p)\n        choices = b & ~(1 << p)\n        assert choices\n        q = (choices & -choices).bit_length()-1\n        second = list(set_bits(b ^ (1 << q)))\n        for i in second:\n            gate('CX',basis[i],basis[q])\n        pivot,*others = target_bits(direction)\n        for out in others:\n            gate('CX',pivot,out)\n        gate('CCX',basis[p],basis[q],pivot)\n        for out in others:\n            gate('CX',pivot,out)\n        for i in reversed(second):\n            gate('CX',basis[i],basis[q])\n        for i in reversed(first):\n            gate('CX',basis[i],basis[p])\n    high_only = [sum(int(ac[h,0,j]) << j for j in range(O)) for h in range(A)]\n    low_only = [sum(int(ac[0,l,j]) << j for j in range(O)) for l in range(B)]\n    low_only[0] = 0\n    def correction(destination,left,right,direction):\n        for a in set_bits(left):\n            for b in set_bits(right):\n                index = (a+1)|(b+1)\n                if index < len(destination):\n                    destination[index] ^= direction\n    for term in terms:\n        pairs = term['factors']\n        direction = term['target_direction']\n        for i in range(0,len(pairs),2):\n            h1,l1 = pairs[i]\n            if i+1 == len(pairs):\n                product(h1,l1 << H,direction)\n            else:\n                h2,l2 = pairs[i+1]\n                # (h1+l2)(h2+l1) + h1*h2 + l1*l2\n                # equals h1*l1 + h2*l2 over GF(2).\n                product(h1 ^ (l2 << H),h2 ^ (l1 << H),direction)\n                correction(high_only,h1,h2,direction)\n                correction(low_only,l1,l2,direction)\n    for q in target_bits(high_only[0]):\n        gate('X',q)\n    for mask in range(1,A):\n        linear_output(high[mask],high_only[mask])\n    for mask in range(1,B):\n        linear_output(low[mask],low_only[mask])\n    for triple in reversed(cache):\n        gate('CCX',*triple)\n    circuit = builder.finish_circuit()\n    native = int(circuit.operation_statistics().num_toffoli_like)\n    assert native == sum(t[0]=='CCX' for t in trace)\n    return circuit,trace,{'num_qubits':int(circuit.num_qubits),\n        'toffoli_count':native,'clean_cached_functions':len(cache),\n        'coherent_cache_compute_and_cleanup_toffolis':2*len(cache),\n        'gates':len(trace)}\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument('--angle-words',type=Path,required=True)\n    p.add_argument('--tree',type=Path,required=True)\n    p.add_argument('--source-factorization',type=Path,required=True)\n    p.add_argument('--input-brot',type=int,default=12)\n    p.add_argument('--brot',type=int,default=8)\n    p.add_argument('--rows',type=int,default=700)\n    p.add_argument('--batch-size',type=int,default=5)\n    p.add_argument('--lowbits',type=int,default=4)\n    p.add_argument('--k',type=int,default=0)\n    p.add_argument('--max-kernels',type=int,default=32767)\n    p.add_argument('--out',type=Path,required=True)\n    p.add_argument('--verifier-dir',type=Path,default=Path('/tmp/chem_verifiers/dfthcsos'))\n    a = p.parse_args()\n    start = time.time()\n    if a.out.exists():\n        raise FileExistsError('refusing to overwrite previous results')\n    original = np.load(a.angle_words,allow_pickle=False)[:a.rows].copy()\n    edges = np.load(a.tree,allow_pickle=False)\n    if a.input_brot < a.brot:\n        raise ValueError('input precision must be at least output precision')\n    shift = a.input_brot-a.brot\n    rounded = ((original+(1 << (shift-1))) >> shift) % (1 << a.brot) if shift else original\n    words,signs = canonical_gauge(rounded,edges,a.brot)\n    gauge_error = float(np.max(np.abs(decode(words,edges,a.brot)-\n                                     signs[:,None]*decode(rounded,edges,a.brot))))\n    assert gauge_error < 1e-12\n    delta = delta_batch(words,a.batch_size,a.k)\n    ac,terms,planning = plan(delta,a.brot,a.lowbits,a.max_kernels)\n    circuit,trace,report = build(delta,a.brot,a.lowbits,ac,terms)\n    report['exact_symbolic_oracle'] = symbolic_check(trace,delta,a.brot,circuit.num_qubits)\n    report['native_oracle'] = native_check(circuit,delta,a.brot,a.verifier_dir)\n    a.out.mkdir(parents=True)\n    np.save(a.out/'fixture_angle_words.npy',words)\n    np.save(a.out/'batch_delta.npy',delta)\n    np.save(a.out/'anf_coefficients.npy',ac)\n    (a.out/'certificate.json').write_text(json.dumps({'plan':planning,'terms':terms},indent=2)+'\\n')\n    cp = a.out/f'angle_k_{a.k}.kmb'\n    circuit.to_file(cp,format='kmb')\n    (a.out/f'angle_k_{a.k}.kmx').write_text(str(circuit))\n    report.update({'scope':'ORGANIZER_SUBSET_CIRCUIT_FIXTURE_ONLY',\n        'chemistry_acceptance_claimed':False,'full_family_compiled':False,\n        'D':len(words),'brot':a.brot,'batch_size':a.batch_size,'k':a.k,\n        'lowbits':a.lowbits,'planning':planning,'gauge_max_coordinate_error':gauge_error,\n        'source_factorization_sha256':sha(a.source_factorization),\n        'source_words_sha256':sha(a.angle_words),'source_tree_sha256':sha(a.tree),\n        'fixture_words_sha256':sha(a.out/'fixture_angle_words.npy'),\n        'compiler_sha256':sha(__file__),'proof_helper_sha256':sha(Path(__file__).with_name('borrowed_prefix.py')),\n        'native_sha256':sha(cp),'seconds':time.time()-start})\n    (a.out/'report.json').write_text(json.dumps(report,indent=2)+'\\n')\n    print(json.dumps({k:v for k,v in report.items() if k!='planning'},indent=2))\n    print(json.dumps({k:v for k,v in planning.items() if k not in ['output_basis_certificates','original_matrix_ranks']},indent=2))\n\n\nif __name__ == '__main__':\n    main()\n"
}
LIGHTNING_AUDITED_SHA256 = {'LICENSE': 'cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30', 'NOTICE': 'dc7c8aba3eee7bcea024f10916dc5a2304b91cff45fca531084665adb866ef2a', 'borrowed_prefix.py': '83410d631e91ed5b1c39c68754cd523d377bbcac621974c77d0a56c7af319a73', 'emit_j5.py': '612b2e90bf83638353a66b7ac07bcec9b018c803c904127f0d493808ff1ffc30', 'extraction.json': '8694b5392d1bad0550d12dad99c2c53173fba5ad41536b211761de5bd6b38fcf', 'hyper_native.py': '131b45c948699d3931ac375b0e67bdbd7aca557b9739da3c2218c41616f15cd9', 'kernel_planes.py': 'eadc1f0c518b9cbfac3752c6b17f476e739c48e830b489fe69551dce49f15550', 'native_checks.py': 'bbd832683b9e380e69cd7fc9437dc61370dbfb0991b9a0b26fd173a00f479eaf', 'permuted_kernel.py': 'bd40dbbbe1a8be409c603aa4e81136b52992bf10e565bbf384fb2ff8f60986c7', 'published-runtime-receipt.json': 'c719f3dd25654f48d730276a805275684f5012ee0ac1f1077d086665ec86a4ad', 'rank_parity.py': '9471a33a3c2907bfccf70acd1ffc5df0104dcee9a891643bbf25cb437626cb5b'}
LIGHTNING_DIR = ROOT/'src_lightningv08'
LIGHTNING_DIR.mkdir(exist_ok=True)
for name, text in LIGHTNING_SOURCE.items():
    path = LIGHTNING_DIR/name
    path.write_text(text)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == LIGHTNING_AUDITED_SHA256[name]
print('Embedded audited files:', sorted(LIGHTNING_SOURCE))


## Runtime and source preflight


Long work runs in a separate, spawn-importable Python module so four-process jobs work from a Kaggle notebook.
The outer notebook owns a process-group watchdog, which also kills descendant verifier workers if the global
deadline expires. Every phase checkpoints JSON. Required-input or API failures stop with an inventory or traceback;
there is no pip, Internet, forked verifier, or fallback training import.


In [ ]:
DRIVER_PARTS.append(r'''"""Fixed-word ANGLE stage A; the notebook writes this spawn-importable module."""
import os
os.environ.update(OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
import collections
import glob
import hashlib
import inspect
import json
import lzma
import math
import multiprocessing as mp
from pathlib import Path
import random
import re
import shutil
import sys
import time
import traceback
import zipfile


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(4 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.partial')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False,
                              default=lambda x: x.item() if hasattr(x, 'item') else str(x)))
    os.replace(tmp, path)


def deep_find(name, root=Path('/kaggle/input')):
    # glob follows Kaggle's symlinked mounts on Python 3.13.
    return sorted({Path(p) for p in glob.glob(str(root / '**' / name), recursive=True)})


def inventory():
    return '\n'.join(sorted(glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
                            + glob.glob('/kaggle/input/*/*/*'))[:100]) or '(no inputs)'


def mounts(slug):
    return [p for p in deep_find(slug.split('/')[-1]) if p.is_dir()]


def public_family(path):
    # Explicit output IDs from manwithacat's public README technique table.
    labels = {'b94b469b148a':'HW-12', '3aac21436c28':'HW-13', '7d372981d3ce':'HW-14'}
    for part in path.parts:
        if part in labels:
            return labels[part]+' ('+part+')'
        if re.fullmatch(r'HW-\d+|open_circuits_v\d+', part):
            return part
    return path.parent.name


def one_hit(name, slug=None):
    hits = deep_find(name)
    if slug:
        hits = [p for p in hits if slug.split('/')[-1] in p.parts]
    if len(hits) != 1:
        raise FileNotFoundError(f'Need one {name} from {slug}; found {hits}\n{inventory()}')
    return hits[0]


def configure(config):
    global C, ROOT, np, km, tq, ej, sc, vc, vi, vd
    C = config
    ROOT = Path(C['work'])
    sys.path[:0] = C['import_paths']
    import numpy as np
    import kickmix as km
    import tensorqrom as tq
    import emit_j5 as ej
    from dfthcsos.scoreinfra import circuits as sc
    from dfthcsos import verify as vc
    from invsqrt import verify_submission_invsqrt as vi
    from dqi import verify_submission_dqi as vd


''')


## Exact tables and scorer accounting


The chemistry archive is unpacked first and verified by the official `v_dfthcsos.verify` at default shots.
`recover_words_from_circuits` cross-checks all 1363×75 words against the NPZ (modulo 256, explicitly reported).
Any real mismatch saves every changed row/column/value and stops this fixed-word experiment.

For batch `B[k]=words[:,5*k:5*k+5]`, use `delta[0]=B[0]`, `delta[k]=B[k] XOR B[k-1]` and assert equality with
`fourqrom.batch_deltas`. Infer affine bit-0 columns from every complete and partial 16-row block, require both adjacent
batches to be coded, and use Lera's own `coded_layout` and `pad_affine` on the last invalid rows. Do not regenerate
rounded words from U or from the tree.

To price an incumbent, the native simulator follows the scorer's actual forward batches and state checks; differences
in its executed `num_toffoli_like` statistic give each `T_k`. Their sum must equal both official
`angles_toffoli_per_pass` and `angles_costs(...)["toffoli_per_pass"]`. `circuit_costs().max_magic` is displayed only as
a static diagnostic. Single-circuit official recovery checks a candidate's exact delta on all addresses, forward and
backward; additional checks initialize the real preceding batch and random signed-gauge targets, as required by the
borrowed top-bit emitter. Measurement repeats are regression checks, not exhaustive enumeration of every branch.


In [ ]:
DRIVER_PARTS.append(r'''# ── Exact tables and scorer instrumentation ──────────────────────────────────
def batch_deltas(words, j=5):
    blocks = [words[:, k:k+j] for k in range(0, words.shape[1], j)]
    assert all(b.shape[1] == j for b in blocks)
    return [b.copy() if k == 0 else b ^ blocks[k-1] for k, b in enumerate(blocks)]


def affine_columns(words):
    """Infer the optional coded metadata discarded by submission.npz."""
    codes = []
    patterns = np.array([[c ^ ((a & m).bit_count() & 1) for a in range(16)]
                         for c in (0, 1) for m in range(16)], dtype=np.int64)
    for j in range(words.shape[1]):
        valid = True
        for start in range(0, len(words), 16):
            bits = words[start:start+16, j] & 1
            if not np.any(np.all(patterns[:, :len(bits)] == bits, axis=1)):
                valid = False
                break
        if valid:
            codes.append(j)
    return codes


def native_forward(circuits, batches, initial=None, seed=7):
    """Same native trajectory and private state checks as the official recovery.

    Read operation_statistics BEFORE/AFTER every forward do: static max_magic
    is only a diagnostic and can overprice classically conditioned instructions.
    """
    layout = sc.register_layout(circuits)
    sc._reject_rotations(circuits)
    repeats, count = sc.SHOTS_PER_ADDRESS, len(batches[0])
    addresses = np.tile(np.arange(count, dtype=np.int64), repeats)
    address_bits = ((addresses[None, :] >> np.arange(len(layout.selection))[:, None]) & 1).astype(bool)
    sim = km.Simulator(batch_size=count*repeats, debug_print=False,
                       seed=int(seed), counting_operations=True)
    sim.use_same_registers_as(circuits[0])
    place = np.int64(1) << np.arange(layout.brot, dtype=np.int64)

    def reset(batch):
        sim.clear_for_shot()
        sim.write_across_shots(layout.selection, address_bits)
        if batch is not None:
            tiled = np.tile(batch, (repeats, 1))
            for j, word in enumerate(layout.words):
                sim.write_across_shots(word, ((tiled[:, j][None, :] >>
                    np.arange(layout.brot)[:, None]) & 1).astype(bool))

    reset(initial)
    costs = []
    for k, circuit in enumerate(circuits):
        before = sim.operation_statistics().num_toffoli_like
        sim.do(circuit)
        costs.append(float(sim.operation_statistics().num_toffoli_like-before)/(count*repeats))
        sc._check_state(sim, circuit, layout, address_bits, f'meter forward {k}')
        actual = np.stack([place @ sim.read_across_shots(w).astype(np.int64)
                           for w in layout.words], axis=1).reshape(repeats, count, layout.batch_size)
        assert np.all(actual == batches[k][None, :, :]), f'forward table mismatch at {k}'
        reset(batches[k])
    return costs


def recover_delta(circuit, count, seed):
    words, layout, both = sc.recover_words_from_circuits(
        [circuit], num_vectors=count, num_angles=5,
        shots_per_address=sc.SHOTS_PER_ADDRESS, seed=int(seed))
    assert layout.brot == 8 and layout.batch_size == 5 and len(layout.selection) == 11
    return words, float(both)/2


def validate_candidate(path, delta, previous, seed):
    circuit = km.Circuit.from_file(str(path), format='kmb')
    assert int(circuit.num_qubits) <= 128
    loaded, average = recover_delta(circuit, len(delta), seed)
    if not np.array_equal(loaded, delta):
        return dict(compatible=False, differing_words=int(np.count_nonzero(loaded != delta)),
                    T_forward_reverse_average=average, Q=int(circuit.num_qubits))
    # Borrowed sign bits require signed-gauge inputs: test the real previous
    # batch, the reverse input, and a random signed-gauge target, all addresses.
    forward = native_forward([circuit], [previous ^ delta], previous, seed+1)[0]
    reverse = native_forward([circuit], [previous], previous ^ delta, seed+2)[0]
    rng = np.random.default_rng(seed)
    target = rng.integers(0, 128, delta.shape, dtype=np.int64)
    target |= ((target >> 6) & 1) << 7
    native_forward([circuit], [target ^ delta], target, seed+3)
    assert abs(forward-average) < 1e-9 and abs(reverse-forward) < 1e-9, 'direction-dependent cost'
    return dict(compatible=True, T=forward, Q=int(circuit.num_qubits), sha256=sha256(path),
                delta_sha256=hashlib.sha256(delta.astype('<i8').tobytes()).hexdigest(),
                addresses=len(delta), repeats=sc.SHOTS_PER_ADDRESS,
                static=sc.circuit_costs([circuit])[0])


def phase_compatible_terms(raw, groups):
    """Convert lightning's nonconstant-mask orientation to Lera's phase cap.

    Preserve legal paired products; split only illegal low forms by phase,
    then pair their rank-one pieces with the same output direction. This is
    an exact adapter, not a claim to reproduce a newer winning search engine.
    """
    terms, loose = [], collections.defaultdict(list)
    singles = sum(1 << (m-1) for m in (1, 2, 4, 8))
    for term in raw:
        try:
            tq._phase(term, groups)
            terms.append(term)
            continue
        except AssertionError:
            pass
        a1, b1, a2, b2, outs = term
        for low, high in ((a1, b1), (a2, b2)):
            remaining = low
            for phase, group in enumerate(groups):
                allowed = sum(1 << (m-1) for m in group) | singles
                part = remaining & allowed
                remaining ^= part
                if part and high:
                    loose[phase, outs].append((part, high))
            assert remaining == 0
    for (phase, outs), pieces in loose.items():
        for i in range(0, len(pieces), 2):
            (a1, b1), (a2, b2) = pieces[i], pieces[i+1] if i+1 < len(pieces) else (0, 0)
            term = (a1, b1, a2, b2, outs)
            assert tq._phase(term, groups) == phase or not (a1 | a2) & ~singles
            terms.append(term)
    return terms


def emit_plan(delta, terms, path, seed, previous):
    tensor = tq.anf_tensor(delta, 8, 4)
    assert np.array_equal(tq.mixed_of(terms, 15, tensor.shape[1]-1, 35), tensor[1:, 1:])
    circuit = tq.build_circuit(delta, terms, 11, 8, 4, groups=tq.low_groups(4), scratch=True)
    circuit.to_file(str(path), format='kmb')
    return validate_candidate(path, delta[:1363], previous, seed)


def synthetic_checks():
    w = np.arange(32*10, dtype=np.int64).reshape(32, 10) % 64
    ds = batch_deltas(w)
    assert np.array_equal(ds[0] ^ ds[1], w[:, 5:])
    coded = np.array([[((a & 3).bit_count() & 1)] for a in range(19)])
    assert affine_columns(coded) == [0]
    coded[7, 0] ^= 1
    assert affine_columns(coded) == []
    raw = [(1 << 2, 3, 1 << 4, 5, 7), (1 << 14, 4, 1 << 1, 3, 8)]
    adapted = phase_compatible_terms(raw, tq.low_groups(4))
    assert np.array_equal(tq.mixed_of(raw, 15, 7, 35), tq.mixed_of(adapted, 15, 7, 35))


''')


## Reproduce the public compilers


Every table receives a Lera control using `shared` defaults (`trials=6`, `seed=k`, `SIZES`/coded grouping) and a
lightningv08 control using its unchanged `emit_j5.best_plan` defaults with a short budget. C40's full cache is known
to need 140 qubits, so the Lera control enters the grouped-cache/scratch branch of `build_family` directly, with the
same decomposition settings and the 128-qubit cap. Workers never call `build_family(jobs=8)`.

Lightning factors are `(high_mask, low_mask)`; Lera terms are `(low1, high1, low2, high2, output_mask)`.
The adapter first proves tensor equality, preserves phase-compatible products, and splits/re-pairs only products that
need more than one low-cache phase. Both raw and adapted counts are saved. This may cost extra Toffolis; a raw rank
count is never advertised as an emitted circuit cost. The report separates correct compilation, matching the incumbent's
cost, no feasible plan, and hard timeout. An incomplete control leaves a stage B HOLD verdict.

Each improving plan certificate includes actual terms, phase groups, output layout, padding, deterministic seed,
delta/source hashes and the serialized/reloaded circuit's native measurement. Hard per-job timeouts terminate processes;
the upstream planner's soft loop is not trusted to interrupt a long inner attempt.


In [ ]:
DRIVER_PARTS.append(r'''# ── Spawn workers: immutable compiler files, bounded independent jobs ─────────
def planner_job(task):
    delta = np.load(task['delta'], allow_pickle=False)
    previous = np.load(task['previous'], allow_pickle=False)
    k, seed = task['k'], task['seed']
    stats = dict(attempts=0, completed_attempts=0, algebra_feasible=0, emitted_feasible=0,
                 reasons={}, certificates=[], best_T=None)
    reasons = collections.Counter()
    started = time.monotonic()
    stop = min(started+task['soft_seconds'], C['search_deadline'])
    out = Path(task['result'])

    def checkpoint():
        stats.update(seconds=time.monotonic()-started, reasons=dict(reasons))
        atomic_json(out, dict(status='RUNNING', k=k, method=task['method'], stats=stats))

    def accept(terms, extra):
        path = Path(task['directory']) / f'candidate_{len(stats["certificates"])}.kmb'
        result = emit_plan(delta, terms, path, seed+stats['attempts'], previous)
        assert result['compatible']
        stats['emitted_feasible'] += 1
        if stats['best_T'] is None or result['T'] < stats['best_T']:
            stats['best_T'] = result['T']
            certificate = dict(k=k, method=task['method'], seed=seed, terms=terms,
                groups=tq.low_groups(4), scratch=True, n_sel=11, brot=8, J=5,
                padded_delta=task['delta'], delta_file_sha256=sha256(task['delta']),
                padding=delta[1363:].tolist(), source_sha256=C['compiler_hashes'],
                circuit=str(path), native=result, **extra)
            cert_path = path.with_suffix('.json')
            atomic_json(cert_path, certificate)
            stats['certificates'].append(str(cert_path))
        checkpoint()

    if task['method'] == 'lera_defaults' or task['method'] == 'lera_search':
        stats['attempts'] += 1
        checkpoint()
        perm, sizes = tq.coded_layout(task['coded_words'], 5, 8)
        tensor = tq.anf_tensor(delta, 8, 4)
        cp = tensor[:, :, perm] if perm is not None else tensor
        terms = tq.shared(cp, sizes, trials=6, seed=seed, groups=tq.low_groups(4))
        if perm is not None:
            terms = [(a1, b1, a2, b2, sum(1 << perm[i] for i in tq._bits(s)))
                     for a1, b1, a2, b2, s in terms]
        stats['completed_attempts'] += 1
        stats['algebra_feasible'] += 1
        accept(terms, dict(output_permutation=perm, group_sizes=sizes, trials=6,
                           control='build_family grouped/scratch branch; no full-cache trial'))
    elif task['method'] == 'lightning_best_plan':
        original = ej.plan_general

        def counted(*args, **kwargs):
            stats['attempts'] += 1
            checkpoint()
            try:
                res, rep = original(*args, **kwargs)
                stats['completed_attempts'] += 1
                stats['algebra_feasible'] += int(res is not None)
                if res is None:
                    reasons[json.dumps(rep, sort_keys=True)] += 1
                checkpoint()
                return res, rep
            except Exception as exc:
                reasons[type(exc).__name__] += 1
                checkpoint()
                raise

        ej.plan_general = counted
        try:
            found = ej.best_plan(delta, budget_seconds=task['soft_seconds'])
        finally:
            ej.plan_general = original
        if found is not None:
            (ac, products), rep = found
            assert np.array_equal(ac.transpose(1, 0, 2), tq.anf_tensor(delta, 8, 4))
            raw = [(int(t['factors'][0][1]), int(t['factors'][0][0]),
                    int(t['factors'][1][1]), int(t['factors'][1][0]), int(t['target_direction']))
                   for t in products]
            terms = phase_compatible_terms(raw, tq.low_groups(4))
            accept(terms, dict(raw_products=len(raw), adapted_products=len(terms),
                               lightning_report=rep, raw_terms=raw))
        else:
            reasons['best_plan returned no feasible plan within soft budget'] += 1
    else:
        rng = random.Random(seed)
        while time.monotonic() < stop:
            stats['attempts'] += 1
            checkpoint()
            perm = list(range(15))
            rng.shuffle(perm)
            rel_base = rng.choice((4, 4, 5, 6, 7))
            try:
                res, rep = ej.plan_general(delta, rel_base=rel_base, extra=15,
                                           seed=rng.randrange(2**30), permutation=perm)
                stats['completed_attempts'] += 1
                if res is None:
                    reasons[json.dumps(rep, sort_keys=True)] += 1
                    checkpoint()
                    continue
                stats['algebra_feasible'] += 1
                ac, products = res
                assert np.array_equal(ac.transpose(1, 0, 2), tq.anf_tensor(delta, 8, 4))
                raw = [(int(t['factors'][0][1]), int(t['factors'][0][0]),
                        int(t['factors'][1][1]), int(t['factors'][1][0]), int(t['target_direction']))
                       for t in products]
                terms = phase_compatible_terms(raw, tq.low_groups(4))
                predicted = tq.cost(terms, 16, len(ac))
                if stats['best_T'] is None or predicted < stats['best_T']:
                    accept(terms, dict(raw_products=len(raw), adapted_products=len(terms),
                                       lightning_report={**rep, 'permutation': perm}, raw_terms=raw))
            except (AssertionError, ValueError) as exc:
                reasons[type(exc).__name__+': '+str(exc)[:180]] += 1
            checkpoint()
    stats.update(seconds=time.monotonic()-started, reasons=dict(reasons))
    return dict(status='COMPLETE', k=k, method=task['method'], stats=stats)


def job_entry(config, task):
    try:
        configure(config)
        if task['method'] == 'public':
            delta = np.load(task['delta'], allow_pickle=False)[:1363]
            prev = np.load(task['previous'], allow_pickle=False)
            result = validate_candidate(task['circuit'], delta, prev, task['seed'])
            value = dict(status='COMPLETE', k=task['k'], method='public', native=result)
        else:
            value = planner_job(task)
        atomic_json(task['result'], value)
    except Exception as exc:
        old = json.loads(Path(task['result']).read_text()) if Path(task['result']).exists() else {}
        atomic_json(task['result'], dict(**{k:v for k,v in old.items() if k not in ('status','error','traceback')},
            status='ERROR', error=repr(exc), traceback=traceback.format_exc()))


def run_jobs(tasks, deadline, callback):
    """Four spawn processes; timeout means terminate/kill, never just future.cancel."""
    ctx = mp.get_context('spawn')
    pending, live = collections.deque(tasks), []
    completed = []
    while pending or live:
        while pending and len(live) < 4 and time.monotonic() < deadline:
            task = pending.popleft()
            Path(task['result']).unlink(missing_ok=True)
            Path(task['result']+'.partial').unlink(missing_ok=True)
            p = ctx.Process(target=job_entry, args=(C, task))
            p.start()
            live.append((p, task, time.monotonic()))
        for item in list(live):
            p, task, start = item
            expired = time.monotonic() >= min(deadline, start+task['hard_seconds'])
            if p.is_alive() and not expired:
                continue
            if p.is_alive():
                p.terminate()
                p.join(2)
                if p.is_alive():
                    p.kill()
            p.join(2)
            result = json.loads(Path(task['result']).read_text()) if Path(task['result']).exists() else {}
            if expired and result.get('status') != 'COMPLETE':
                result.update(status='TIMEOUT')
            result.update(k=task['k'], method=task['method'], wall_seconds=time.monotonic()-start,
                          task=task, exitcode=p.exitcode)
            completed.append(result)
            callback(result)
            live.remove(item)
        if time.monotonic() >= deadline and not live:
            break
        time.sleep(.15)
    for task in pending:
        result = dict(status='NOT_STARTED_BUDGET', k=task['k'], method=task['method'], task=task)
        completed.append(result)
        callback(result)
    return completed


''')


## Public portfolio compatibility


The recursive finder handles root `angle_k_*.kmb`, chemistry hash directories, and flattened
`open_circuits_v*/dfthcsos__angle_k_*.kmb` names. It scans the optional datasets, keeps every path as provenance, and
deduplicates byte-identical files before native simulation. A family label is attached to an incumbent only when the
actual circuit SHA-256 matches. HW labels are retained when present in paths; historical evidence is not used to guess
a byte identity. Shape, λ, a claimed T, and a matching factorization are insufficient compatibility tests.

Every checked candidate must decode our exact delta on all 1363 valid addresses, preserve selection and clean scratch,
and fit 128 qubits. All compatible lower-than-incumbent circuits are collected, even when another candidate is cheaper.
Missing inputs and untested/time-limited candidates remain visible in `public_coverage`; partial coverage is never
reported as a complete best-of search.


In [ ]:
DRIVER_PARTS.append(r'''# ── Main research phases and conditional full packaging ──────────────────────
def main(config):
    configure(config)
    report = dict(experiment='07_dfthcsos_angle_loader_search_A', status='RUNNING',
                  phase='preflight', config=C, reproduction=[], public_candidates=[], jobs=[],
                  missing_optional=[], baseline=None, per_k=[], packaged=False)
    summary = ROOT / 'loader_search_A.json'
    started = time.monotonic()

    def save():
        report['elapsed_seconds'] = time.monotonic()-started
        atomic_json(summary, report)

    def phase(name):
        report['phase'] = name
        print(f'[{(time.monotonic()-started)/60:.1f} min] {name}', flush=True)
        save()

    try:
        synthetic_checks()
        assert sc.SHOTS_PER_ADDRESS == 64 and vc.QUBIT_CAP == 128
        for module, function, args in ((sc, 'recover_words_from_circuits', ('num_vectors','num_angles','seed')),
                                       (tq, 'shared', ('trials','seed','groups')),
                                       (tq, 'build_circuit', ('groups','scratch')),
                                       (ej, 'best_plan', ('budget_seconds',))):
            assert all(a in inspect.signature(getattr(module, function)).parameters for a in args)
        assert 'max_workers' in inspect.signature(vi.verify).parameters
        assert 'max_workers' in inspect.signature(vd.verify).parameters
        manifest_path = Path(C['manifest'])
        manifest = json.loads(manifest_path.read_text())
        entries = {item['file']: item for item in manifest['files']}
        expected = {'invsqrt.kmb', 'dqi.kmb', 'dfthcsos/submission.npz',
                    *(f'dfthcsos/angle_k_{k}.kmb' for k in range(15))}
        assert set(entries) == expected, f'release layout changed: {set(entries) ^ expected}'
        report['release'] = dict(dataset='heliosli/qcc-circuits-live', label=manifest.get('portfolio'),
            manifest_sha256=sha256(manifest_path), manifest=manifest,
            declared_sha256={name:item['sha256'] for name,item in entries.items()}, stored_sha256={})
        atomic_json(ROOT / 'pinned_release_manifest.json', manifest)
        print('Pinned release:', manifest.get('portfolio'), '(expected at authoring:', C['expected_label'], ')', flush=True)
        public = ROOT / 'pinned_public'
        chemistry = public / 'dfthcsos'
        chemistry.mkdir(parents=True, exist_ok=True)

        def unpack(name):
            item = entries[name]
            src = manifest_path.parent / item['stored_as']
            if not src.is_file():
                hits = deep_find(Path(item['stored_as']).name, manifest_path.parent)
                assert len(hits) == 1, (name, hits)
                src = hits[0]
            dst = public / name
            dst.parent.mkdir(parents=True, exist_ok=True)
            report['release']['stored_sha256'][item['stored_as']] = sha256(src)
            if src.suffix == '.xz':
                with lzma.open(src) as fi, dst.open('wb') as fo:
                    shutil.copyfileobj(fi, fo, 4 << 20)
            else:
                shutil.copyfile(src, dst)
            assert sha256(dst) == item['sha256'], f'{name}: release sha256 mismatch'
            assert dst.stat().st_size == item['bytes'], f'{name}: byte size mismatch'
            print(item['sha256'], name, flush=True)
            return dst

        for name in sorted(expected):
            if name.startswith('dfthcsos/'):
                unpack(name)
        report['source_sha256'] = dict(C['compiler_hashes'])
        verifier_files = sorted((Path(C['competition']) / 'verifiers').rglob('*.py'))
        report['verifier_sha256_before'] = {str(p):sha256(p) for p in verifier_files}
        report['kickmix_sha256'] = {str(p):sha256(p) for p in (Path(C['competition'])/'kickmix').rglob('*')
                                    if p.is_file() and p.suffix in ('.so','.py','.pyd')}
        phase('official baseline chemistry, default 64 shots/address')
        t0 = time.monotonic()
        baseline = vc.verify(str(chemistry), salt=C['salt'])
        report['baseline_seconds'] = time.monotonic()-t0
        report['baseline'] = baseline
        assert baseline['status'] == 'PASS' and baseline['num_qubits'] == 428
        assert baseline['brot'] == 8 and baseline['batch_size'] == 5 and baseline['num_circuits'] == 15
        assert baseline['resources']['heuristic_toffoli'] == 6741
        assert baseline['verified_shots'] == 1363*64
        phase('recover words, forward executed costs, delta/code reconstruction')
        with np.load(chemistry/'submission.npz', allow_pickle=False) as z:
            raw_words = z['angle_words'].copy()
            words = raw_words.astype(np.int64) % 256
            assert words.shape == (1363,75) and z['tree_edges'].shape == (75,2)
            assert z['W'].shape == (9,143,40)
        family = sc.load_circuits(str(chemistry), format='kmb')
        recovered, layout, both = sc.recover_words_from_circuits(
            family, num_vectors=1363, num_angles=75, seed=C['seed'], shots_per_address=64)
        mismatch = np.argwhere(recovered != words)
        report['word_crosscheck'] = dict(exact=np.array_equal(recovered, raw_words),
            modulo_256_equal=not len(mismatch), representation='npz integers reduced modulo 256',
            differing_words=len(mismatch))
        if len(mismatch):
            atomic_json(ROOT/'word_mismatches.json', [dict(row=int(a), column=int(b),
                npz=int(words[a,b]), circuit=int(recovered[a,b])) for a,b in mismatch])
            # The verifier scores the words the circuits load, not the NPZ copy (SCORING_README: angle_words in the
            # NPZ is an extra the verifier does not read). manwithacat's HW-14 changed thirteen dead words in the
            # circuits only, so the circuit words are the ground truth for every table below.
            print(f'NOTE: {len(mismatch)} words differ between NPZ and circuits (expected: HW-14 dead-word edits); '
                  'using the circuit-loaded words', flush=True)
            report['word_crosscheck']['source'] = 'circuits (verifier ground truth)'
            words = recovered.astype(np.int64) % 256
        assert not np.any(((words >> 7) ^ (words >> 6)) & 1)
        from fourqrom import batch_deltas as official_deltas
        deltas = batch_deltas(words)
        assert np.array_equal(np.asarray(deltas), np.asarray(official_deltas(words, 5)))
        costs = native_forward(family, [words[:,k*5:k*5+5] for k in range(15)], seed=C['seed']+1)
        assert math.isclose(sum(costs), both/2, abs_tol=1e-9)
        assert math.isclose(sum(costs), baseline['resources']['angles_toffoli_per_pass'], abs_tol=1e-9)
        assert vc.angles_costs(family, both)['toffoli_per_pass'] == sum(costs)
        coded = set(affine_columns(words))
        report['coded_columns'] = sorted(coded)
        tables = ROOT/'angle_tables'
        tables.mkdir(exist_ok=True)
        best = []
        for k, delta in enumerate(deltas):
            cw = [j for j in range(5) if 5*k+j in coded and (k == 0 or 5*(k-1)+j in coded)]
            padded = tq.pad_affine(delta, cw) if cw else delta.copy()
            prev = np.zeros_like(delta) if k == 0 else words[:,(k-1)*5:k*5]
            dp, pp = tables/f'delta_{k}.npy', tables/f'previous_{k}.npy'
            np.save(dp, padded, allow_pickle=False)
            np.save(pp, prev, allow_pickle=False)
            row = dict(k=k, incumbent_T=costs[k], incumbent_Q=int(family[k].num_qubits),
                incumbent_sha256=sha256(chemistry/f'angle_k_{k}.kmb'), incumbent_families=[],
                delta_sha256=hashlib.sha256(delta.astype('<i8').tobytes()).hexdigest(),
                coded_words=cw, delta=str(dp), previous=str(pp), static=sc.circuit_costs([family[k]])[0])
            report['per_k'].append(row)
            best.append(dict(T=costs[k], Q=int(family[k].num_qubits), path=str(chemistry/f'angle_k_{k}.kmb'),
                             source='pinned heliosli release', certificate=None))
        del family

        def task_for(k, method, seed, soft, hard, index):
            row = report['per_k'][k]
            directory = ROOT/'planner_jobs'/f'{index:05d}_{method}_k{k}_s{seed}'
            directory.mkdir(parents=True, exist_ok=True)
            return dict(k=k, method=method, seed=seed, delta=row['delta'], previous=row['previous'],
                        coded_words=row['coded_words'], soft_seconds=soft, hard_seconds=hard,
                        result=str(directory/'result.json'), directory=str(directory))

        def ingest(result, bucket):
            report[bucket].append(result)
            if bucket == 'reproduction':
                row = report['per_k'][result['k']]
                value = result.get('stats',{}).get('best_T')
                row[result['method']+'_T'] = value
                row[result['method']+'_status'] = result['status']
                row[result['method']+'_gap_to_incumbent'] = None if value is None else value-row['incumbent_T']
            for cert in result.get('stats',{}).get('certificates',[]):
                plan = json.loads(Path(cert).read_text())
                measured = plan['native']
                k = plan['k']
                if measured['compatible'] and measured['T'] < best[k]['T'] and measured['Q'] <= 128:
                    best[k] = dict(T=measured['T'], Q=measured['Q'], path=plan['circuit'],
                                   source=plan['method'], certificate=cert)
                    print('Improved', k, '->', measured['T'], plan['method'], flush=True)
            for k,row in enumerate(report['per_k']):
                row['best'] = best[k]
            save()

        phase('reproduction: Lera shared defaults, then published lightning best_plan')
        tasks = [task_for(k, 'lera_defaults', k, C['lera_seconds']-15, C['lera_seconds'], k)
                 for k in range(15)]
        tasks += [task_for(k, 'lightning_best_plan', k, C['lightning_seconds'], C['lightning_seconds']+20, 15+k)
                  for k in range(15)]
        repro_stop = min(time.monotonic()+60*C['reproduction_minutes'], C['search_deadline'])
        run_jobs(tasks, repro_stop, lambda r:ingest(r,'reproduction'))
        repro = {}
        for method in ('lera_defaults','lightning_best_plan'):
            rows = [r for r in report['reproduction'] if r['method'] == method]
            repro[method] = dict(covered=len(rows), complete=sum(r['status']=='COMPLETE' for r in rows),
                emitted=sum(bool(r.get('stats',{}).get('certificates')) for r in rows),
                cost_matches_incumbent=sum(r.get('stats',{}).get('best_T')==costs[r['k']] for r in rows),
                status='REPRODUCED' if len(rows)==15 and all(r['status']=='COMPLETE' and
                       r.get('stats',{}).get('certificates') for r in rows) else 'INCOMPLETE_OR_NO_PLAN')
        report['reproduction_status'] = repro
        report['reproduction_explanation'] = (
            'Lera uses the exact grouped branch of build_family (SIZES/coded_layout, trials=6, seed=k); '
            'the impossible 140-qubit full cache is skipped. Lightning uses unchanged best_plan defaults '
            'and records raw vs phase-adapted products. Different current incumbent counts are expected '
            'from unpublished extended-relation searches and HW-14 word edits; no equality of engines is claimed. '
            'Timeout or no feasible plan is unresolved and is not evidence of successful reproduction.')

        phase('public best-of: decode every unique attached loader against our exact deltas')
        candidate_files = []
        for slug in C['optional_datasets']:
            roots = mounts(slug)
            if not roots:
                report['missing_optional'].append(slug)
                print('MISSING OPTIONAL:', slug, '\n'+inventory(), flush=True)
            for root in roots:
                for p in deep_find('*angle_k_*.kmb', root):
                    match = re.fullmatch(r'(?:dfthcsos__)?angle_k_(\d+)\.kmb', p.name)
                    if match and int(match[1]) < 15:
                        candidate_files.append((slug, p, int(match[1])))
        # Hash every path, preserve all aliases, simulate each byte-identical
        # circuit only once. Identification by matching bytes, never by T alone.
        aliases = collections.defaultdict(list)
        for slug,p,k in candidate_files:
            digest = sha256(p)
            aliases[k,digest].append(dict(dataset=slug, path=str(p), family=public_family(p)))
            if digest == report['per_k'][k]['incumbent_sha256']:
                report['per_k'][k]['incumbent_families'].append(dict(dataset=slug,path=str(p),family=public_family(p)))
        public_tasks = []
        for index, ((k,digest), sources) in enumerate(sorted(aliases.items())):
            task = task_for(k, 'public', C['seed']+index, 30, C['public_job_seconds'], 1000+index)
            task.update(circuit=sources[0]['path'], aliases=sources, sha256=digest)
            public_tasks.append(task)

        def public_ingest(result):
            task = result['task']
            native = result.get('native',{})
            k = result['k']
            if native.get('compatible') and native['T'] < report['per_k'][k]['incumbent_T']:
                result['compatible_lower_than_incumbent'] = True
                if native['T'] < best[k]['T']:
                    best[k] = dict(T=native['T'], Q=native['Q'], path=task['circuit'],
                                   source=task['aliases'], certificate=None)
            ingest(result, 'public_candidates')

        public_stop = min(time.monotonic()+60*C['public_minutes'], C['search_deadline'])
        run_jobs(public_tasks, public_stop, public_ingest)
        report['public_coverage'] = dict(paths=len(candidate_files), unique=len(public_tasks),
            checked=sum(r['status']=='COMPLETE' for r in report['public_candidates']),
            complete=not report['missing_optional'] and all(r['status']=='COMPLETE' for r in report['public_candidates']),
            missing_optional=report['missing_optional'])

''')


## Throughput calibration, official verification and packaging


For 25 wall minutes, four independently seeded spawn jobs cover all tables round-robin. They alternate the reusable
kernel/quotient planner (relation budgets and low-monomial permutations) and Lera's default relation heuristic with new
seeds. This is a calibration of the available public Python planners, not the unavailable Rust HW-14 or newer winning
matroid search. The report records started/completed attempts, native/algebra feasibility, timeouts, worker- and wall-minute
rates, and best `T_k`. Imports, emission and checks are included in measured job rates.

If anything improves, certificates are re-emitted and all winning binaries reloaded; only the corresponding ANGLE files
replace the pinned chemistry files. `submission.npz` remains byte-identical. Full official chemistry verification must
PASS with `Q=428`, fixed `lambda_eff`, and `T=6741+2*sum(T_k)`. Only then, with at least the configured 30-minute
reserve remaining (never below 25), unpack invsqrt/DQI and run their official default-shot verifiers with four workers.
The 18-member ZIP is written atomically after every member is rehashed and the verifier source hashes are unchanged.

No saving means no ZIP and no unpacking of invsqrt/DQI. In that branch, the chemistry ratio is measured but GM is null:
the other ratios have no fresh verified run. A timeout also cannot leave a submission ZIP behind. The final JSON and
printed verdict distinguish reproduction status, search coverage, measured savings, full verification and stage B advice.


In [ ]:
DRIVER_PARTS.append(r'''        phase('25-minute four-process throughput calibration')
        calibration_start = time.monotonic()
        calibration_stop = min(calibration_start+60*C['calibration_minutes'], C['search_deadline'])
        index = 10000
        round_id = 0
        while time.monotonic() < calibration_stop-2:
            # Round robin guarantees all 15 tables get a first job before repeats.
            tasks = []
            for k in range(15):
                method = 'lightning_search' if round_id % 2 == 0 else 'lera_search'
                tasks.append(task_for(k, method, C['seed']+100*round_id+k,
                    C['calibration_job_seconds']-10, C['calibration_job_seconds'], index))
                index += 1
            run_jobs(tasks, calibration_stop, lambda r:ingest(r,'jobs'))
            round_id += 1
        calibration_wall = time.monotonic()-calibration_start
        throughput = []
        for k in range(15):
            rows = [r for r in report['jobs'] if r['k']==k]
            attempts = sum(r.get('stats',{}).get('attempts',0) for r in rows)
            completed_attempts = sum(r.get('stats',{}).get('completed_attempts',0) for r in rows)
            feasible = sum(r.get('stats',{}).get('algebra_feasible',0) for r in rows)
            emitted = sum(r.get('stats',{}).get('emitted_feasible',0) for r in rows)
            seconds = sum(r.get('wall_seconds',0) for r in rows)
            throughput.append(dict(k=k, jobs=len(rows), attempts_started=attempts,
                attempts_completed=completed_attempts, attempts_per_worker_minute=60*completed_attempts/seconds if seconds else 0,
                attempts_per_wall_minute=60*completed_attempts/calibration_wall if calibration_wall else 0,
                feasible_rate=feasible/completed_attempts if completed_attempts else None,
                algebra_feasible=feasible, native_feasible=emitted,
                best_T=best[k]['T'], timeouts=sum(r['status']=='TIMEOUT' for r in rows)))
        report['throughput'] = dict(wall_seconds=calibration_wall, processes=4, start_method='spawn',
            per_k=throughput, note='Rates include import, planning, emission and native checks; aborted attempts are separate.')
        report['best_total_T'] = 6741+2*sum(x['T'] for x in best)
        report['chemistry_ratio'] = C['reference_cost']['dfthcsos']/(baseline['lambda_eff']*428*report['best_total_T'])
        report['expected_GM'] = None
        report['other_ratios'] = None
        saved_pass = sum(costs)-sum(x['T'] for x in best)
        reproduced = all(v['status']=='REPRODUCED' for v in repro.values())
        report['stage_B_recommendation'] = (
            'GO: a measured native improvement and all controls reproduced; consider 12 h with observed throughput.'
            if saved_pass > 0 and reproduced else
            'HOLD: resolve reproduction/phase/timeout gaps before a 12 h search.' if not reproduced else
            'NO-GO for seed-only 12 h search: controls reproduced but this calibration found no saving; improve search strategy first.')
        report['saved_per_pass'] = saved_pass
        report['lb_arithmetic'] = [dict(saved_per_pass=s,
            gain_at_15777_and_24_17645=24.17645*((15777/(15777-2*s))**(1/3)-1)) for s in (10,20,60)]
        save()
        if saved_pass > 0:
            phase('assemble chemistry with byte-identical npz and exact reloaded winners')
            submit = ROOT/'submission_A'
            if submit.exists():
                shutil.rmtree(submit)
            shutil.copytree(chemistry, submit/'dfthcsos')
            for k,item in enumerate(best):
                if item['T'] < costs[k]:
                    # Re-emit certificates rather than trusting only a saved binary.
                    if item['certificate']:
                        cert = json.loads(Path(item['certificate']).read_text())
                        temp = ROOT/'planner_jobs'/f'final_k{k}.kmb'
                        emit_plan(np.load(cert['padded_delta'],allow_pickle=False), cert['terms'], temp,
                                  C['seed']+k, np.load(report['per_k'][k]['previous'],allow_pickle=False))
                        item['path'] = str(temp)
                    check = validate_candidate(item['path'], deltas[k],
                        np.load(report['per_k'][k]['previous'],allow_pickle=False), C['seed']+1000+k)
                    assert check['compatible'] and check['T'] == item['T']
                    shutil.copyfile(item['path'], submit/'dfthcsos'/f'angle_k_{k}.kmb')
            assert sha256(submit/'dfthcsos/submission.npz') == entries['dfthcsos/submission.npz']['sha256']
            final_chem = vc.verify(str(submit/'dfthcsos'), salt=C['salt'])
            assert final_chem['status']=='PASS' and final_chem['num_qubits']==428
            assert final_chem['toffoli_count']==6741+2*sum(x['T'] for x in best)
            assert final_chem['lambda_eff']==baseline['lambda_eff']
            report['final_chemistry'] = final_chem
            save()
            if C['hard_deadline']-time.monotonic() < 60*C['verification_reserve_minutes']:
                report['packaging_skip'] = f'Less than {C["verification_reserve_minutes"]}-minute verification reserve remains: no three-task ZIP.'
                report['GM_unavailable_reason'] = 'Full invsqrt/DQI verification skipped for insufficient reserve.'
            else:
                phase('unpack invsqrt/DQI; full official verifiers and atomic 18-member ZIP')
                for name in ('invsqrt.kmb','dqi.kmb'):
                    shutil.copyfile(unpack(name), submit/name)
                results = dict(dfthcsos=final_chem)
                t0 = time.monotonic()
                circuit = km.Circuit.from_file(str(submit/'invsqrt.kmb'),format='kmb')
                results['invsqrt'] = vi.verify(circuit, vi.ProblemSpecInvSqrt.for_n_p(10),
                                               salt=C['salt'], max_workers=4)
                report['invsqrt_seconds'] = time.monotonic()-t0
                del circuit
                t0 = time.monotonic()
                circuit = km.Circuit.from_file(str(submit/'dqi.kmb'),format='kmb')
                results['dqi'] = vd.verify(circuit, vd.ProblemSpecDQI.benchmark_instance(),
                                           salt=C['salt'], max_workers=4)
                report['dqi_seconds'] = time.monotonic()-t0
                del circuit
                assert all(r['status']=='PASS' for r in results.values())
                report['verified_results'] = results
                ratios = {task:C['reference_cost'][task]/float(r['score']) for task,r in results.items()}
                report['other_ratios'] = {task:ratios[task] for task in ('invsqrt','dqi')}
                report['expected_GM'] = math.prod(ratios.values())**(1/3)
                files = sorted(p.relative_to(submit).as_posix() for p in submit.rglob('*') if p.is_file())
                assert set(files)==expected and len(files)==18
                hashes = {name:sha256(submit/name) for name in files}
                for name in ('invsqrt.kmb','dqi.kmb','dfthcsos/submission.npz'):
                    assert hashes[name]==entries[name]['sha256']
                partial = ROOT/'submission.zip.partial'
                with zipfile.ZipFile(partial,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
                    for name in files:
                        z.write(submit/name,name)
                with zipfile.ZipFile(partial) as z:
                    assert sorted(z.namelist())==files
                    for name in files:
                        h = hashlib.sha256()
                        with z.open(name) as f:
                            for chunk in iter(lambda:f.read(4 << 20),b''):
                                h.update(chunk)
                        assert h.hexdigest()==hashes[name]
                report['verifier_sha256_after'] = {str(p):sha256(p) for p in verifier_files}
                assert report['verifier_sha256_before']==report['verifier_sha256_after']
                provenance = dict(release=report['release'], sha256=hashes, per_k=report['per_k'],
                    compiler_sha256=C['compiler_hashes'], verifier_sha256=report['verifier_sha256_after'],
                    results=results, expected_GM=report['expected_GM'], salt=C['salt'], default_shots=True)
                atomic_json(ROOT/'provenance.json',provenance)
                os.replace(partial,ROOT/'submission.zip')
                report['packaged'] = True
                report['submission_zip_sha256'] = sha256(ROOT/'submission.zip')
        else:
            report['packaging_skip'] = 'No compatible lower-T loader: report only; invsqrt/DQI not unpacked or verified.'
            report['GM_unavailable_reason'] = 'Other task ratios require a fresh full verifier run; no published receipts substituted.'
        report['verifier_sha256_after'] = {str(p):sha256(p) for p in verifier_files}
        assert report['verifier_sha256_after']==report['verifier_sha256_before']
        report['status'] = 'COMPLETE'
        report['phase'] = 'done'
        for k,row in enumerate(report['per_k']):
            row['best'] = best[k]
        save()
    except BaseException as exc:
        report.update(status='FAILED', error=repr(exc), traceback=traceback.format_exc())
        save()
        (ROOT/'submission.zip').unlink(missing_ok=True)
        raise
    finally:
        print('Report:', summary, flush=True)


if __name__ == '__main__':
    main(json.loads(Path(sys.argv[1]).read_text()))
''')


In [ ]:
DRIVER_PATH = ROOT/'loader_stage_a.py'
DRIVER_PATH.write_text(''.join(DRIVER_PARTS))
ast.parse(DRIVER_PATH.read_text())
sys.path.insert(0, str(ROOT))
import loader_stage_a as stage_a
try:
    release_manifest = stage_a.one_hit('QCC_RELEASE_MANIFEST.json', 'heliosli/qcc-circuits-live')
    invsqrt_verify = stage_a.one_hit('verify_submission_invsqrt.py')
    competition = invsqrt_verify.parent.parent.parent
    assert (competition/'verifiers/dfthcsos/verify.py').is_file()
    tensor_source = stage_a.one_hit('tensorqrom.py', 'marculera/dfthc-compact-pipeline')
    fourqrom_hits = stage_a.deep_find('fourqrom.py', tensor_source.parent)
    if not fourqrom_hits:
        fourqrom_hits = stage_a.deep_find('fourqrom.py', tensor_source.parent.parent)
    assert len(fourqrom_hits) == 1, ('Ambiguous/missing fourqrom.py', fourqrom_hits)
    fourqrom_source = fourqrom_hits[0]
    compiler_hashes = {str(tensor_source):stage_a.sha256(tensor_source),
                       str(fourqrom_source):stage_a.sha256(fourqrom_source),
                       **{str(LIGHTNING_DIR/name):digest for name,digest in LIGHTNING_AUDITED_SHA256.items()},
                       str(DRIVER_PATH):stage_a.sha256(DRIVER_PATH)}
    print('Lera source hashes:', compiler_hashes[str(tensor_source)], compiler_hashes[str(fourqrom_source)])
    AUDITED_LERA_SHA256 = {'tensorqrom.py': 'e6bbcaf1ad7ad025f2105f0233aa9e232a211d1d76bac867da9282b21d4ed017', 'fourqrom.py': '8421c051b6b86e93bfd23722a416e7aa8aee5b4eaf7693a8cd61eb3bce3fc968'}
    for source in (tensor_source, fourqrom_source):
        if stage_a.sha256(source) != AUDITED_LERA_SHA256[source.name]:
            print('NOTE: compiler differs from audited snapshot; record bytes and require API/tensor/native checks:', source)
    RUN_CONFIG = dict(work=str(ROOT), competition=str(competition), manifest=str(release_manifest),
        import_paths=[str(competition/'kickmix'), str(competition/'verifiers'), str(tensor_source.parent),
                      str(fourqrom_source.parent), str(LIGHTNING_DIR)],
        optional_datasets=OPTIONAL_DATASETS, expected_label=EXPECTED_PORTFOLIO,
        compiler_hashes=compiler_hashes, seed=SEED, salt=SALT, reference_cost=REFERENCE_COST,
        hard_deadline=HARD_DEADLINE,
        search_deadline=HARD_DEADLINE-(VERIFICATION_RESERVE_MINUTES+5)*60,
        verification_reserve_minutes=VERIFICATION_RESERVE_MINUTES,
        reproduction_minutes=REPRODUCTION_MINUTES, lera_seconds=LERA_HARD_SECONDS,
        lightning_seconds=LIGHTNING_SOFT_SECONDS, public_minutes=PUBLIC_MINUTES,
        public_job_seconds=PUBLIC_JOB_SECONDS, calibration_minutes=CALIBRATION_MINUTES,
        calibration_job_seconds=CALIBRATION_JOB_SECONDS)
    CONFIG_PATH = ROOT/'loader_search_A_config.json'
    stage_a.atomic_json(CONFIG_PATH, RUN_CONFIG)
except Exception as exc:
    stage_a.atomic_json(ROOT/'loader_search_A.json', dict(status='PREFLIGHT_FAILED', error=repr(exc),
        required_competition=REQUIRED_COMPETITION, required_datasets=REQUIRED_DATASETS,
        inventory=stage_a.inventory(), packaged=False))
    print(stage_a.inventory())
    raise


In [ ]:
# One process group contains the driver, four planners and all verifier children.
# Its wall deadline includes notebook setup; cleanup has another <10 seconds.
LOG = ROOT/'loader_search_A.log'
with LOG.open('w') as output:
    process = subprocess.Popen([sys.executable, '-u', str(DRIVER_PATH), str(CONFIG_PATH)],
        stdout=output, stderr=subprocess.STDOUT, start_new_session=True, cwd=ROOT)
offset = 0
timed_out = False
try:
    while process.poll() is None:
        if time.monotonic() >= HARD_DEADLINE:
            timed_out = True
            break
        with LOG.open() as f:
            f.seek(offset)
            chunk = f.read()
            offset = f.tell()
        if chunk:
            print(chunk, end='', flush=True)
        time.sleep(1)
finally:
    if process.poll() is None:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            pass
        # Kill descendants even if the group leader exited on SIGTERM.
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait(timeout=3)
with LOG.open() as f:
    f.seek(offset)
    print(f.read(), end='', flush=True)
if timed_out or process.returncode != 0:
    path = ROOT/'loader_search_A.json'
    interrupted = json.loads(path.read_text()) if path.exists() else {}
    interrupted.update(status='GLOBAL_TIMEOUT' if timed_out else 'FAILED',
                       driver_exitcode=process.returncode, packaged=False)
    stage_a.atomic_json(path, interrupted)
    (ROOT/'submission.zip').unlink(missing_ok=True)
    (ROOT/'submission.zip.partial').unlink(missing_ok=True)
    print('No submission ZIP: inspect loader_search_A.json and loader_search_A.log')


## Verdict

`REPRODUCED` means that every table was compiled, emitted and natively checked with that public control. It does
not mean the current incumbent's unpublished search was reproduced. Compare each default's actual cost gap, then
raw/adapted lightning terms and failure reasons before deciding whether 12 hours is justified. A measured best-of
saving is useful even if the available planners are slower. Expected GM is populated only from this run's full official
three-task results; otherwise it remains unavailable rather than borrowing a published number.


In [ ]:
summary = json.loads((ROOT/'loader_search_A.json').read_text())
print('STATUS:', summary['status'], '| phase:', summary.get('phase'), '| packaged:', summary.get('packaged'))
print('RELEASE:', summary.get('release',{}).get('label'))
print('REPRODUCTION:', json.dumps(summary.get('reproduction_status',{}), indent=2))
print('k | incumbent | Lera defaults | lightning adapted | best | Q | source')
for row in summary.get('per_k',[]):
    best = row.get('best',{})
    print(row['k'], row['incumbent_T'], row.get('lera_defaults_T'), row.get('lightning_best_plan_T'),
          best.get('T'), best.get('Q'), best.get('source'), sep=' | ')
    print('  incumbent byte-identical families:', row.get('incumbent_families') or 'unidentified in attached inputs')
print('TOTAL T:', summary.get('best_total_T'), '| chemistry ratio:', summary.get('chemistry_ratio'))
print('Other ratios (fresh verified run):', summary.get('other_ratios'))
print('Expected GM:', summary.get('expected_GM'), summary.get('GM_unavailable_reason',''))
print('PUBLIC COVERAGE:', summary.get('public_coverage'))
print('THROUGHPUT:', json.dumps(summary.get('throughput',{}), indent=2))
print('STAGE B:', summary.get('stage_B_recommendation', 'Resolve failed preflight/run first.'))
print('PACKAGING:', summary.get('packaging_skip', '/kaggle/working/submission.zip' if summary.get('packaged') else 'none'))
print('Artifacts:', ROOT/'loader_search_A.json', '|', LOG)
if summary.get('error'):
    print('ERROR:', summary['error'])


# Sources and credits

- **Lera / marculera** — the compact C40 factorization, words, code layout and public tensorqrom compiler:
  [Hamiltonian notebook v10](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian/version/10)
  and [dfthc-compact-pipeline dataset](https://www.kaggle.com/datasets/marculera/dfthc-compact-pipeline).
- **mrmeows** — exact paired-product lookup, cached monomials and HMR cleanup design, as credited in Lera's emitter.
- **Ahmed Berat Özer** — generic kernel-relation emitter underlying the reusable planner, Apache-2.0.
- **Vladislav Lavrentev / lightningv08** — [Bezout-Schur + 5-Word QROM v15](https://www.kaggle.com/code/lightningv08/bezout-schur-5-word-qrom/version/15),
  reusable planner sources embedded byte-for-byte with Apache-2.0 LICENSE and NOTICE.
- **matousakura** — [extended relations and portfolio v12](https://www.kaggle.com/code/matousakura/best-of-public-portfolio-vqcc-24-0x-class/version/12);
  [public circuit dataset](https://www.kaggle.com/datasets/matousakura/best-of-public-portfolio).
- **James Barlow / manwithacat** — [HW-12/13/14 searches and dead-word edits, v15](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook/version/15),
  [Hedge Wizardry dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-hedge-wizardry),
  [Apache public circuits](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits).
- **Dmitrii Gluzdov** — [admission, matroid-parity and address-basis prior art, v49](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source/version/49),
  [factorization/public circuits dataset](https://www.kaggle.com/datasets/dmitriigluzdov/vqcc-femoco-public-factorization).
  The inspected dataset does not provide the latest search engine; this notebook does not claim to reproduce it.
- **heliosli** — [live release, selected weights and portfolio provenance](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live);
  current manifest label and all declared hashes are captured before any changes.
- **SomeJoint / Agniv Banerjee's team** — coded angle bits; **Kozykorpesh / kozykappa and Sumit Chongder**, and
  **Artem1410 / anazemcev** — signed/banked lookup lineage, as credited upstream.

The full invsqrt/DQI lineage below is copied from `01_public_portfolio.ipynb`:

- **invsqrt** — Ehan Mallick, [invsqrt 20.65: two 3-to-2 AND merges on y6225](https://www.kaggle.com/code/ehanmallick/invsqrt-20-65-two-3-to-2-and-merges-on-y6225)
  (two GF(2) quadratic-form windows where three ANDs become two), on yoyobar and GPT-6 Astra's 62-qubit circuit
  ([LB 22.99 | invsqrt 101 - 62 qubits](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits)), as shipped
  by Alex Chilton. Earlier lineage as the authors credit it: mrmeows / team selenium
  ([21.0 circuits](https://www.kaggle.com/code/mrmeows/21-0-circuits-let-s-all-reach-the-plateau)) → Akmal Xodarev
  (busyaprime) 63 qubits → yoyobar's cone rewrites → Ahmed Berat Özer's phase correction (as distributed in manwithacat's
  [VQCC Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo)); earlier still Phuc Thai (DePie),
  Dmitrii Gluzdov, heliosli and marsyas.

- **dqi** — yoyobar's 3 November single paid-gate product-cache reuse (453,309 paid Toffoli-equivalents), on his earlier
  lead-product reuse and caching, on Akmal Xodarev's (busyaprime)
  [Exact DQI Rewrites at 892 Qubits](https://www.kaggle.com/code/busyaprime/lb-23-91003-exact-dqi-rewrites-at-892-qubits)
  (flag clears by X-basis measurement, symbolic AND caching, interval allocation), which uses manwithacat's two-wire
  clean ladder ([Hedge Wizardry HW-07](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook)) and
  Ahmed Berat Özer's product-caching idea, on SomeJoint's (Agniv Banerjee's team)
  [892-qubit decoder](https://www.kaggle.com/code/somejoint/quantum-circuits-24-15-local-gm-892-qubit-dqi). SomeJoint
  credits GPT-6 Astra Pro, Claude, Ali Haydar Özdağ (flag carry), yoyobar (one-carry history) and the earlier decoder
  line: jday96314 (QuantumAI), Alex Chilton, mrmeows, manwithacat, Dmitrii Gluzdov, ViberTeam and Phuc Thai (DePie).

- **Organizers** — Google Quantum AI (Tanuj Khattar, Nicholas C. Rubin, Craig Gidney, Michael Broughton, Guang Hao Low,
  Ryan Babbush), [arXiv:2610.23832](https://arxiv.org/abs/2610.23832), competition verifiers and native kickmix.
  The attached `dfthcsos/verify.py` and `scoreinfra/circuits.py` define the authoritative API, word recovery, register
  contract, and executed-gate costs; their files are hashed before and after verification.
- **This notebook** — fixed-word compatibility scan, phase adapter, per-loader native cost metering, bounded calibration
  and reproducible certificates. It adds no factorization, word edits, training stack or winning-search claim.

Licences remain with their respective authors. The embedded source LICENSE/NOTICE accompany the runtime artifacts.
Planning estimates on four Kaggle CPU cores: preflight/unpack 1–5 min; baseline chemistry about 30 s;
word recovery and forward accounting 1–3 min; reproduction 5–20 min; optional public scan 0–25 min;
calibration 25 min; winner checks/chemistry 1–5 min; conditional invsqrt about 750 s, DQI about 170 s, ZIP 1–4 min.
Actual timings and timeouts, not these estimates, drive the verdict. The 148-minute group watchdog and reserve checks
take precedence. Local checks were restricted to JSON/AST validation and tiny synthetic NumPy algebra tests.
